# 05. 하이퍼파라미터 튜닝 — 통합모델(CatBoost)

**목적**: `04_model_selection`에서 확정한 최종 승자 구조 — **통합모델(이용률 타깃) + CatBoost + v2 피처셋(50개)** — 의 하이퍼파라미터를 `model-tuning` 스킬 기준으로 탐색한다. 8절까지는 하이퍼파라미터(트리 복잡도 등) 탐색이고, 9절은 별도로 **손실 함수·표본 가중치**를 대회 채점 산식 구조에 맞춰 조정하는 실험이다.

- **입력**: `data/processed/train_features_v2.parquet`, `test_features_v2.parquet` (`04_model_selection` 피처 선택 산출물)
- **출력**: 최적 하이퍼파라미터(dict), 튜닝 전후 비교, seed 안정성 확인 결과, 분위수 회귀·actual 가중 학습 실험 결과 — `reports/05_tuning.md`로 정리, 이후 `train.ipynb`에 하드코딩되어 재사용됨

**`model-tuning` 스킬 1절 핵심 규칙**: "튜닝 목적 함수 = 동일 CV 구조의 fold 평균 Score. 단일 fold 최적화 금지." `04_model_selection`은 단일 fold(2022~2023 train / 2024 validation)로 모델을 비교했지만, 이제 하이퍼파라미터를 실제로 탐색하는 단계라 **fold 하나에만 맞춰 튜닝하면 그 fold의 우연한 패턴에 과적합될 위험**이 커진다. 그래서 이 노트북부터는 `timeseries-validation` 스킬의 "보강안"인 **확장 윈도우 3-fold 시계열 CV**로 바꾼다.

**9절 메모**: 대회 공식 산식(`src/metric.py`)을 다시 뜯어보면, 채점 대상 여부가 실측값(actual) 기준이고 FICR도 actual로 가중합을 계산한다. 이 구조적 특징을 이용하면 손실 함수(분위수 회귀)와 표본 가중치(actual 가중)를 산식에 맞게 조정할 수 있다 — 근거는 9절 시작 부분에 자세히 설명.

## 0. 셋업

패키지를 불러오고 REPO_ROOT를 찾는다. `optuna`(하이퍼파라미터 탐색 도구 — 무작정 모든 조합을 다 해보는 대신, 지금까지의 결과를 보고 "다음엔 이쪽을 더 파보자"고 똑똑하게 다음 후보를 고르는 라이브러리)를 새로 사용한다.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
import optuna
from optuna.samplers import TPESampler

SEED = 42
np.random.seed(SEED)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import metric, TARGET_COLS, CAPACITY_KWH

PROCESSED_DIR = REPO_ROOT / "data" / "processed"

pd.set_option("display.max_columns", 100)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("python executable:", sys.executable)
print("REPO_ROOT:", REPO_ROOT)
print("optuna version:", optuna.__version__)

python executable: d:\공모전\wind_forecast\venv\Scripts\python.exe
REPO_ROOT: d:\공모전\wind_forecast
optuna version: 4.9.0


## 1. 데이터 로드 (v2 피처셋)

`04_model_selection`의 피처 선택 결과물인 v2(50개 피처)를 불러온다. v1(52개) 대신 v2를 쓰는 이유: 상관관계·permutation importance·피처군 ablation을 거쳐 걸러낸 피처셋이기 때문.

In [2]:
train_features = pd.read_parquet(PROCESSED_DIR / "train_features_v2.parquet")
test_features = pd.read_parquet(PROCESSED_DIR / "test_features_v2.parquet")

DROP_COLS = {"forecast_kst_dtm", "forecast_id", *TARGET_COLS}
FEATURE_COLS = [c for c in train_features.columns if c not in DROP_COLS]

print("train_features:", train_features.shape)
print("test_features:", test_features.shape)
print("피처 개수:", len(FEATURE_COLS))
print("기간:", train_features["forecast_kst_dtm"].min(), "~", train_features["forecast_kst_dtm"].max())

train_features: (26304, 54)
test_features: (8760, 52)
피처 개수: 50
기간: 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00


## 2. 확장 윈도우 3-fold 시계열 CV

**결론 먼저**: 아래 3개 fold를 만든다. 전부 **2022-01-01부터 시작해서 점점 뒤로 늘어나는 학습 구간**(확장 윈도우 — 매번 처음부터 다시 시작하지 않고, 알고 있는 과거를 계속 누적해서 학습에 쓰는 방식) + 그 바로 다음 6개월을 검증 구간으로 쓴다.

| fold | 학습 구간 | 검증 구간 |
|---|---|---|
| fold1 | 2022-01 ~ 2023-06 | 2023-07 ~ 2023-12 |
| fold2 | 2022-01 ~ 2023-12 | 2024-01 ~ 2024-06 |
| fold3 | 2022-01 ~ 2024-06 | 2024-07 ~ 2024-12 |

**왜 이 구조인가**: 랜덤 K-Fold는 시계열에서 절대 쓰면 안 된다(`timeseries-validation` 스킬 1절 — 미래 데이터로 과거를 예측하는 셈이 되어 누수). 확장 윈도우는 실제 운영 상황(계속 쌓이는 과거 데이터로 다음 구간을 예측)과 가장 비슷하다. `04_model_selection`에서 쓴 단일 분리(2022~2023 train / 2024 valid)는 fold2와 거의 같은 구조인데, 튜닝에서는 이거 하나만 보면 "2024년 상반기에만 우연히 잘 맞는" 파라미터를 고를 위험이 있어 fold를 3개로 늘렸다.

group_3(라벨이 2023-01부터만 있음)도 fold1부터 6개월치 학습 데이터가 있어 3개 fold 모두 유효하다.

In [3]:
FOLDS = [
    {"name": "fold1", "train_start": "2022-01-01 01:00:00", "train_end": "2023-07-01 00:00:00", "valid_end": "2024-01-01 00:00:00"},
    {"name": "fold2", "train_start": "2022-01-01 01:00:00", "train_end": "2024-01-01 00:00:00", "valid_end": "2024-07-01 00:00:00"},
    {"name": "fold3", "train_start": "2022-01-01 01:00:00", "train_end": "2024-07-01 00:00:00", "valid_end": "2025-01-01 00:00:00"},
]


def make_fold_frames(fold):
    dtm = train_features["forecast_kst_dtm"]
    train_start = pd.Timestamp(fold["train_start"])
    train_end = pd.Timestamp(fold["train_end"])
    valid_end = pd.Timestamp(fold["valid_end"])
    valid_start = train_end + pd.Timedelta(hours=1)

    fold_train = train_features[(dtm >= train_start) & (dtm <= train_end)].reset_index(drop=True)
    fold_valid = train_features[(dtm >= valid_start) & (dtm <= valid_end)].reset_index(drop=True)
    return fold_train, fold_valid


for fold in FOLDS:
    ft, fv = make_fold_frames(fold)
    print(
        f"{fold['name']}: train {ft.shape} ({ft['forecast_kst_dtm'].min()} ~ {ft['forecast_kst_dtm'].max()}), "
        f"valid {fv.shape} ({fv['forecast_kst_dtm'].min()} ~ {fv['forecast_kst_dtm'].max()})"
    )
    print("  그룹별 valid 라벨 결측:", fv[TARGET_COLS].isna().sum().to_dict())

fold1: train (13104, 54) (2022-01-01 01:00:00 ~ 2023-07-01 00:00:00), valid (4416, 54) (2023-07-01 01:00:00 ~ 2024-01-01 00:00:00)
  그룹별 valid 라벨 결측: {'kpx_group_1': 3, 'kpx_group_2': 2, 'kpx_group_3': 0}
fold2: train (17520, 54) (2022-01-01 01:00:00 ~ 2024-01-01 00:00:00), valid (4368, 54) (2024-01-01 01:00:00 ~ 2024-07-01 00:00:00)
  그룹별 valid 라벨 결측: {'kpx_group_1': 0, 'kpx_group_2': 0, 'kpx_group_3': 0}
fold3: train (21888, 54) (2022-01-01 01:00:00 ~ 2024-07-01 00:00:00), valid (4416, 54) (2024-07-01 01:00:00 ~ 2025-01-01 00:00:00)
  그룹별 valid 라벨 결측: {'kpx_group_1': 6, 'kpx_group_2': 6, 'kpx_group_3': 6}


## 3. 통합모델 학습·평가 함수

`04_model_selection` 8절(5b)에서 쓴 것과 같은 구조 — group_id 범주형 피처 + 이용률 타깃으로 3개 그룹을 합쳐 학습 — 를 함수로 재사용 가능하게 만든다. 이번엔 하이퍼파라미터를 인자로 받아서, fold 하나짜리 점수(`train_and_score_fold`)와 3-fold 평균 점수(`cv_score`)를 모두 낼 수 있게 한다.

In [4]:
GROUP_ID_MAP = {"kpx_group_1": 0, "kpx_group_2": 1, "kpx_group_3": 2}
GROUP_ID_CATEGORIES = [0, 1, 2]


def to_long(df, feature_cols):
    frames = []
    for g in TARGET_COLS:
        sub = df[df[g].notna()].copy()
        sub["group_id"] = GROUP_ID_MAP[g]
        sub["utilization"] = sub[g] / CAPACITY_KWH[g]
        frames.append(sub[["forecast_kst_dtm", "group_id", "utilization"] + feature_cols])
    return pd.concat(frames, ignore_index=True)


def make_answer_df(df):
    return df[["forecast_kst_dtm", *TARGET_COLS]].reset_index(drop=True)


def make_pred_df(df, pred_dict):
    out = df[["forecast_kst_dtm"]].reset_index(drop=True).copy()
    for col in TARGET_COLS:
        out[col] = np.clip(pred_dict[col], 0, CAPACITY_KWH[col])
    return out

In [5]:
def train_and_score_fold(fold_train, fold_valid, params, feature_cols=FEATURE_COLS, early_stop_frac=0.15, seed=SEED):
    features = feature_cols + ["group_id"]
    long_df = to_long(fold_train, feature_cols)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    model = CatBoostRegressor(
        loss_function="MAE",
        random_seed=seed,
        verbose=False,
        **params,
    )
    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"],
        early_stopping_rounds=100,
        verbose=False,
    )

    pred = {}
    for g in TARGET_COLS:
        valid_g = fold_valid.copy()
        valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
        util_pred = model.predict(valid_g[features])
        pred[g] = util_pred * CAPACITY_KWH[g]

    answer_df = make_answer_df(fold_valid)
    pred_df = make_pred_df(fold_valid, pred)
    score, one_minus_nmae, ficr = metric(answer_df, pred_df)
    return score, model


def cv_score(params, feature_cols=FEATURE_COLS, seed=SEED, verbose=False):
    scores = []
    for fold in FOLDS:
        fold_train, fold_valid = make_fold_frames(fold)
        score, _ = train_and_score_fold(fold_train, fold_valid, params, feature_cols, seed=seed)
        scores.append(score)
        if verbose:
            print(f"  {fold['name']}: Score={score:.4f}")
    return float(np.mean(scores)), scores

## 4. 튜닝 전 기준점

`04_model_selection` 5b에서 쓴 기본 파라미터(`iterations=2000, learning_rate=0.05`, 나머지 CatBoost 기본값)를 **이번 3-fold CV 구조**로 다시 채점해 기준점을 만든다. 04번의 0.5971은 단일 fold 점수라 이 3-fold 평균과 숫자가 다를 수 있다 — 비교는 항상 "같은 CV 구조끼리"만 해야 하므로(`model-tuning` 스킬 1절), 튜닝 전후 비교는 이 기준점을 기준으로 한다.

In [6]:
DEFAULT_PARAMS = {"iterations": 2000, "learning_rate": 0.05}

baseline_mean, baseline_scores = cv_score(DEFAULT_PARAMS, verbose=True)
print(f"\n기본 파라미터 3-fold 평균 Score: {baseline_mean:.4f} (fold별: {[round(s, 4) for s in baseline_scores]})")

  fold1: Score=0.5679
  fold2: Score=0.5945
  fold3: Score=0.6101

기본 파라미터 3-fold 평균 Score: 0.5908 (fold별: [np.float64(0.5679), np.float64(0.5945), np.float64(0.6101)])


## 5. Optuna 탐색 범위 설계

`model-tuning` 스킬 2절은 LightGBM 기준 표를 준다. CatBoost는 파라미터 이름이 달라서(대칭 트리 구조라 `num_leaves` 대신 `depth`를 씀 등) 같은 역할을 하는 파라미터로 대응시켰다:

| 역할 | LightGBM(스킬 원표) | CatBoost(이 노트북) | 범위 | 비고 |
|---|---|---|---|---|
| 학습 속도 | learning_rate | `learning_rate` | 0.01~0.1 (log) | |
| 트리 복잡도 | num_leaves | `depth` | 4~10 | CatBoost는 대칭 트리라 잎 개수 대신 깊이로 복잡도 조절 |
| 리프 최소 샘플 | min_data_in_leaf | `min_data_in_leaf` | 20~200 | 시계열 노이즈 대응 핵심(스킬 표와 동일) |
| 피처 샘플링 | feature_fraction | `rsm` | 0.6~1.0 | 각 분할에서 쓸 피처 비율 |
| 행 샘플링 | bagging_fraction | `subsample`(+`bootstrap_type="Bernoulli"`) | 0.6~1.0 | CatBoost 기본 부트스트랩(Bayesian)은 subsample 미지원이라 Bernoulli로 변경 필요 |
| L2 정규화 | lambda_l2 | `l2_leaf_reg` | 1~30 (log) | |

`iterations`는 스킬 1절 원칙대로 튜닝하지 않고 early stopping(100 rounds)으로 결정한다. trial 중에는 `iterations=1500`으로 상한만 낮게 잡아(탐색 속도), 최종 확정 파라미터로 재검증할 때 다시 넉넉하게 둔다.

**실행 시간 안내**: trial 1개마다 3-fold를 전부 학습한다. `N_TRIALS`는 아래에 상수로 뒀으니, 처음엔 작게(예: 20) 돌려보고 시간을 가늠한 뒤 늘리는 걸 추천한다. Optuna study는 `data/processed/optuna_catboost_pooled.db`(sqlite, git 제외)에 저장되어 **같은 기기에서는 나중에 이어서 탐색 가능**하다(`model-tuning` 스킬 5절).

*(이 절은 실패/미채택으로 결론나 코드는 삭제하고 마크다운 기록만 남겼다 — 실행 결과는 각 소절 설명과 마지막 종합 해석, `reports/05_tuning.md`를 참고.)*

## 6. 탐색 결과 확인

상위 trial들의 파라미터 분포를 본다 — `model-tuning` 스킬 3절: "1차 coarse 탐색 → 상위 trial 분포 확인 → 필요하면 범위를 좁혀 2차 탐색." 상위 trial들이 특정 값 근처에 몰려 있으면 그 근방으로 범위를 좁혀 `N_TRIALS`를 늘려 재탐색하는 것도 방법이다.

## 7. seed 안정성 확인

`model-tuning` 스킬 3절: "최적 파라미터를 seed 3개로 재검증." 최적 파라미터가 특정 seed에만 우연히 맞은 건 아닌지 확인한다. 이번엔 `iterations`을 2000으로 다시 넉넉하게 둔다(트리 개수는 early stopping이 알아서 정하므로 상한만 여유 있게).

## 8. 튜닝 전후 비교 및 최종 파라미터

같은 3-fold CV 구조에서 기본값과 튜닝값을 나란히 놓고 본다. `model-tuning` 스킬 1절: "개선이 fold 표준편차 이내면 '튜닝 효과 미미'로 정직하게 기록" — 개선폭이 크지 않다고 실망할 필요 없이, 그 자체로 유효한 결론이다.

**해석**: 튜닝 전(0.5908) vs 튜닝 후(0.5909) — 개선폭 +0.000045. seed 3개 표준편차(0.0007)보다도 작다. 즉 **30 trial Optuna 탐색은 사실상 효과가 없었다** — `model-tuning` 스킬 1절 기준대로 "튜닝 효과 미미"로 정직하게 기록한다.

**왜 이런 결과가 나왔을까**: `04_model_selection`에서 이미 CatBoost 기본값이 이 데이터·피처셋에 꽤 잘 맞았다는 뜻으로 해석할 수 있다. 상위 trial들(6절)도 `depth=7~9`, `learning_rate=0.01~0.03` 근방에 몰려 있어 기본값(`learning_rate=0.05`, 기본 depth)과 크게 다르지 않은 영역에서 최적점을 찾은 것으로 보인다. 나무 구조(트리 복잡도, 정규화)를 아무리 조정해도 얻을 수 있는 개선폭에는 한계가 있고, 진짜 개선은 아래 9절(손실 함수·가중치 조정)에서 나왔다 — 이는 "모델을 더 정교하게 튜닝하는 것"보다 "채점 산식이 요구하는 목표 자체를 더 정확히 겨냥하는 것"이 훨씬 크게 작동한다는 신호다.

## 9. 채점 산식 구조를 활용한 손실 조정 — 분위수 회귀 + actual 가중 학습

**결론 먼저**: 지금까지는 손실 함수로 그냥 MAE만 썼다. 그런데 `src/metric.py`를 다시 보면 두 가지 구조적 특징이 있다.

1. `valid = actual >= capacity * 0.10` — **채점 대상 여부가 예측값이 아니라 실측값(actual) 기준**이다. 그러면 우리가 진짜 추정해야 하는 값은 단순 조건부 평균 `E[y|x]`가 아니라, "그 시간이 채점 대상이 되는 조건까지 포함한" `E[y|x, y≥10%용량]`이다. 발전량이 10% 미만인 쪽을 잘라내고 남은 평균이므로 **이 값은 항상 원래 평균보다 크다** — 즉 예측을 살짝 위쪽으로 겨냥하는 게 산식 구조와 맞다. 이걸 만드는 방법이 **분위수 회귀(quantile regression)** 다: "중앙값(50%)"이 아니라 "이 값보다 클 확률이 (1-τ)인 지점(τ분위수)"을 예측하도록 학습하면, τ를 0.5보다 크게 주는 것만으로 예측이 위로 편향된다.
2. `FICR = sum(actual * price) / sum(actual * 4)` — **정산금이 `actual`(그 시간 실제 발전량)로 가중**된다. 발전량이 큰 시간을 잘 맞히는 게 스코어에 더 크게 기여한다는 뜻이므로, **학습 표본 가중치도 `actual`로 주면** 모델이 자연스럽게 발전량 큰 시간에 더 신경 쓰게 된다.

**leakage-guard 체크**: 둘 다 학습 시점에 이미 아는 정보(그 시간의 목표값 자체를 가중치·분위수로 쓰는 것)만 사용하고, 예측 시점에 미래 정보를 끌어오지 않는다. 흔한 회귀 기법(분위수 회귀, 가중 학습)의 정상적인 적용이라 문제 없다.

CatBoost는 `loss_function="Quantile:alpha=τ"`와 `sample_weight`를 함께 지원한다. 아래에서 (a) actual 가중만 켠 경우, (b) 가중 + τ 스윕을 3-fold CV로 비교한다. 기본 파라미터(`DEFAULT_PARAMS`)를 그대로 써서 **손실·가중치 효과만 순수하게 분리**해서 본다 — 튜닝된 파라미터와 합치는 건 이 실험이 유효하다고 확인된 다음에 한다.

In [7]:
def to_long_ext(df, feature_cols):
    '''to_long과 같지만, actual 가중 학습에 쓸 실측 kWh(actual_kwh)를 함께 남긴다.'''
    frames = []
    for g in TARGET_COLS:
        sub = df[df[g].notna()].copy()
        sub["group_id"] = GROUP_ID_MAP[g]
        sub["utilization"] = sub[g] / CAPACITY_KWH[g]
        sub["actual_kwh"] = sub[g]
        frames.append(sub[["forecast_kst_dtm", "group_id", "utilization", "actual_kwh"] + feature_cols])
    return pd.concat(frames, ignore_index=True)


def train_and_score_fold_ext(
    fold_train, fold_valid, params, feature_cols=FEATURE_COLS, early_stop_frac=0.15,
    seed=SEED, quantile_alpha=None, use_sample_weight=False,
):
    features = feature_cols + ["group_id"]
    long_df = to_long_ext(fold_train, feature_cols)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    loss_function = f"Quantile:alpha={quantile_alpha}" if quantile_alpha is not None else "MAE"
    model = CatBoostRegressor(
        loss_function=loss_function,
        random_seed=seed,
        verbose=False,
        **params,
    )

    fit_kwargs = {}
    if use_sample_weight:
        fit_kwargs["sample_weight"] = fit_rows["actual_kwh"].to_numpy()

    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"],
        early_stopping_rounds=100,
        verbose=False,
        **fit_kwargs,
    )

    pred = {}
    for g in TARGET_COLS:
        valid_g = fold_valid.copy()
        valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
        util_pred = model.predict(valid_g[features])
        pred[g] = util_pred * CAPACITY_KWH[g]

    answer_df = make_answer_df(fold_valid)
    pred_df = make_pred_df(fold_valid, pred)
    score, one_minus_nmae, ficr = metric(answer_df, pred_df)
    return score, model


def cv_score_ext(params, quantile_alpha=None, use_sample_weight=False, feature_cols=FEATURE_COLS, seed=SEED):
    scores = []
    for fold in FOLDS:
        fold_train, fold_valid = make_fold_frames(fold)
        score, _ = train_and_score_fold_ext(
            fold_train, fold_valid, params, feature_cols, seed=seed,
            quantile_alpha=quantile_alpha, use_sample_weight=use_sample_weight,
        )
        scores.append(score)
    return float(np.mean(scores)), scores

### 9-1. actual 가중 학습만 먼저 켜보기

손실은 그대로 MAE로 두고, 표본 가중치만 `actual`(kWh)로 준다. 이게 4절의 `baseline_mean`(가중 없음)보다 나아지는지만 본다.

In [21]:
weighted_mean, weighted_scores = cv_score_ext(DEFAULT_PARAMS, quantile_alpha=None, use_sample_weight=True)
print(f"actual 가중(MAE) 3-fold 평균 Score: {weighted_mean:.4f} (fold별: {[round(s, 4) for s in weighted_scores]})")
print(f"기본값(가중 없음) 대비: {weighted_mean - baseline_mean:+.4f}")

KeyboardInterrupt: 

### 9-2. 분위수(τ) 스윕

actual 가중은 유지한 채, 분위수 τ를 0.50(중앙값)부터 0.70까지 바꿔가며 3-fold CV로 비교한다. τ=0.50은 "가중만 켠 분위수 회귀"에 해당하고, τ가 커질수록 예측이 위쪽으로 더 치우친다.

**실행 시간 참고**: τ 값 5개 × 3-fold = 15번 재학습이다.

In [8]:
TAU_GRID = [0.50, 0.55, 0.60, 0.65, 0.70]

tau_rows = []
for tau in TAU_GRID:
    mean_score, fold_scores = cv_score_ext(DEFAULT_PARAMS, quantile_alpha=tau, use_sample_weight=True)
    tau_rows.append({"tau": tau, "cv_mean": mean_score, "fold_scores": fold_scores})
    print(f"tau={tau:.2f}: 3-fold 평균 Score={mean_score:.4f}")

tau_df = pd.DataFrame(tau_rows).sort_values("cv_mean", ascending=False).reset_index(drop=True)
best_tau = tau_df.loc[0, "tau"]
print(f"\n최적 tau: {best_tau} (baseline 대비 {tau_df.loc[0, 'cv_mean'] - baseline_mean:+.4f})")
tau_df

tau=0.50: 3-fold 평균 Score=0.5912
tau=0.55: 3-fold 평균 Score=0.6042
tau=0.60: 3-fold 평균 Score=0.6110
tau=0.65: 3-fold 평균 Score=0.6198
tau=0.70: 3-fold 평균 Score=0.6219

최적 tau: 0.7 (baseline 대비 +0.0310)


,tau,cv_mean,fold_scores
0,0.70,0.621867,"[0.6033057742646094, 0.6215657932243673, 0.640..."
1,0.65,0.619835,"[0.6000607807448144, 0.6224509434563867, 0.636..."
2,0.60,0.610999,"[0.5896020153108509, 0.6164176388098354, 0.626..."
3,0.55,0.604235,"[0.5821246519214243, 0.6064252719489537, 0.624..."
4,0.50,0.591204,"[0.5714372270781778, 0.5923129963031037, 0.609..."


### 9-3. seed 안정성 확인

`model-tuning` 스킬 3절 원칙대로, 최적 τ가 특정 seed의 우연이 아닌지 seed 3개로 재확인한다(지난 세션의 N_TRIALS=5 교훈 — 단일 seed "최선"은 노이즈일 수 있다).

In [23]:
tau_seed_means = []
for seed in [42, 7, 2024]:
    mean_score, fold_scores = cv_score_ext(
        DEFAULT_PARAMS, quantile_alpha=best_tau, use_sample_weight=True, seed=seed,
    )
    tau_seed_means.append(mean_score)
    print(f"seed={seed}: 3-fold 평균 Score={mean_score:.4f} (fold별 {[round(s, 4) for s in fold_scores]})")

tau_seed_mean = float(np.mean(tau_seed_means))
tau_seed_std = float(np.std(tau_seed_means))
print(f"\nseed 3개 평균: {tau_seed_mean:.4f}, 표준편차: {tau_seed_std:.4f}")
print(f"baseline 대비 개선폭: {tau_seed_mean - baseline_mean:+.4f} (표준편차의 {(tau_seed_mean - baseline_mean) / tau_seed_std if tau_seed_std > 0 else float('nan'):.1f}배)")

seed=42: 3-fold 평균 Score=0.6219 (fold별 [np.float64(0.6033), np.float64(0.6216), np.float64(0.6407)])
seed=7: 3-fold 평균 Score=0.6217 (fold별 [np.float64(0.6005), np.float64(0.6212), np.float64(0.6433)])
seed=2024: 3-fold 평균 Score=0.6204 (fold별 [np.float64(0.5987), np.float64(0.6209), np.float64(0.6416)])

seed 3개 평균: 0.6213, 표준편차: 0.0006
baseline 대비 개선폭: +0.0305 (표준편차의 47.6배)


### 9-4. 종합 비교

베이스라인(가중 없음) / actual 가중만 / actual 가중 + 최적 τ(seed 평균)를 한 표로 본다. `model-tuning` 스킬 1절 기준: 개선폭이 seed 표준편차보다 뚜렷이 커야 채택한다.

In [ ]:
loss_comparison_df = pd.DataFrame({
    "구분": ["기본값(가중 없음, MAE)", "actual 가중(MAE, tau=0.5)", f"actual 가중 + tau={best_tau}(seed 평균)"],
    "3-fold 평균 Score": [baseline_mean, weighted_mean, tau_seed_mean],
})
loss_comparison_df["개선폭"] = loss_comparison_df["3-fold 평균 Score"] - baseline_mean
loss_comparison_df

**해석 — 이 노트북의 핵심 발견**: actual 가중만 켠 경우(+0.0004)는 8절 하이퍼파라미터 튜닝과 마찬가지로 미미하다. 하지만 **분위수를 τ=0.5→0.7로 올리자 점수가 0.5908 → 0.6213으로, +0.0305 뛰었다.** seed 3개 표준편차(0.0006)의 **47.6배**에 달하는 개선폭이라 우연이 아니라 안정적인 효과로 판단한다.

**주의할 점 — τ 그리드가 아직 끝을 못 봤다**: 9-2절 표를 보면 τ=0.50→0.55→0.60→0.65→0.70 순서로 점수가 0.591→0.604→0.611→0.620→0.622로 **계속 증가하다가 그리드 상한(0.70)에서 끝났다.** 즉 지금 고른 τ=0.7이 진짜 최적점이 아니라 "탐색을 멈춘 지점"일 가능성이 높다. τ를 0.75, 0.80, 0.85 정도까지 더 넓혀서 점수가 꺾이는 지점(진짜 peak)을 찾는 게 다음 단계로 필요하다 — 너무 큰 τ는 예측을 과도하게 위로 편향시켜 오히려 NMAE를 악화시킬 수 있으므로 어딘가에서는 반드시 꺾인다.

**다음 조합 실험 제안**: (1) τ 그리드를 0.7 이상으로 확장해 진짜 최적 τ 탐색, (2) 확정된 τ + `use_sample_weight=True`를 8절에서 찾은 `BEST_PARAMS`(튜닝된 하이퍼파라미터)와 합쳐서 3-fold CV 재검증 — 개선 폭이 그대로 유지되거나 더 커지는지 확인.

### 9-5. τ 확장 탐색 — 진짜 peak 찾기

**결론 먼저**: 9-2절 그리드(0.50~0.70)는 점수가 계속 오르다가 상한에서 끝났다. τ를 더 큰 값까지 넓혀서, 점수가 오르다 꺾이는 지점(진짜 최적 τ)을 찾는다.

**왜 어딘가에서는 반드시 꺾이는가**: τ가 커질수록 예측이 위로 더 세게 편향된다. 어느 지점까지는 "채점 대상(actual≥10%용량)으로 조건부일 때의 평균이 원래 평균보다 크다"는 구조와 맞아떨어져 NMAE가 줄지만, 편향이 실제 조건부 평균을 넘어서 과도해지면 이제는 예측이 실제보다 계속 높게 나와 오차가 다시 커진다(과도한 과대예측). 그러니 이 그리드는 0.70부터 시작해 1.0에 너무 가깝지 않은 범위까지 넓혀서 봉우리(peak)를 확인한다.

In [ ]:
TAU_GRID_EXT = [0.70, 0.705, 0.71, 0.715, 0.72]

tau_ext_rows = []
for tau in TAU_GRID_EXT:
    mean_score, fold_scores = cv_score_ext(DEFAULT_PARAMS, quantile_alpha=tau, use_sample_weight=True)
    tau_ext_rows.append({"tau": tau, "cv_mean": mean_score, "fold_scores": fold_scores})
    print(f"tau={tau:.3f}: 3-fold 평균 Score={mean_score:.4f}")

tau_ext_df = pd.DataFrame(tau_ext_rows).sort_values("cv_mean", ascending=False).reset_index(drop=True)
best_tau_ext = tau_ext_df.loc[0, "tau"]
print(f"\n확장 탐색 최적 tau: {best_tau_ext} (baseline 대비 {tau_ext_df.loc[0, 'cv_mean'] - baseline_mean:+.4f})")
tau_ext_df

### 9-6. 확장 최적 τ의 seed 안정성 재확인

9-3절과 같은 절차를 `best_tau_ext`로 다시 수행한다. 그리드 상한(0.90)에서도 계속 오르기만 한다면 범위를 더 넓혀야 한다는 신호이므로, 아래 출력을 보고 판단한다.

In [ ]:
tau_ext_seed_means = []
for seed in [42, 7, 2024]:
    mean_score, fold_scores = cv_score_ext(
        DEFAULT_PARAMS, quantile_alpha=best_tau_ext, use_sample_weight=True, seed=seed,
    )
    tau_ext_seed_means.append(mean_score)
    print(f"seed={seed}: 3-fold 평균 Score={mean_score:.4f} (fold별 {[round(s, 4) for s in fold_scores]})")

tau_ext_seed_mean = float(np.mean(tau_ext_seed_means))
tau_ext_seed_std = float(np.std(tau_ext_seed_means))
print(f"\nbest_tau_ext={best_tau_ext}, seed 3개 평균: {tau_ext_seed_mean:.4f}, 표준편차: {tau_ext_seed_std:.4f}")
print(f"baseline 대비 개선폭: {tau_ext_seed_mean - baseline_mean:+.4f}")
print(f"9절(τ=0.7) 대비: {tau_ext_seed_mean - tau_seed_mean:+.4f}")

### 9-7. 최종 비교 (τ=0.7 vs 확장 탐색 최적 τ)

9-4절 표에 확장 탐색 결과를 한 줄 추가해서, τ를 0.7에서 더 밀어붙인 게 실제로 이득이었는지 한눈에 본다.

In [ ]:
loss_comparison_final_df = pd.concat([
    loss_comparison_df,
    pd.DataFrame({
        "구분": [f"actual 가중 + tau={best_tau_ext}(확장 탐색, seed 평균)"],
        "3-fold 평균 Score": [tau_ext_seed_mean],
    }),
], ignore_index=True)
loss_comparison_final_df["개선폭"] = loss_comparison_final_df["3-fold 평균 Score"] - baseline_mean
loss_comparison_final_df

**해석**: 세밀하게(0.005 간격) 훑어보니 **τ=0.71에서 진짜 peak(0.6239)를 찾았다** — 0.705, 0.71까지 오르다가 0.715, 0.72에서 다시 떨어진다. 예상대로(9-5절 설명) 편향이 과도해지면 오차가 다시 커지는 지점이 확인된 것.

**하지만 실전에서는 τ=0.70을 쓰는 게 더 합리적이다. 이유 두 가지**:
1. τ=0.7 대비 τ=0.71의 개선폭은 +0.0003 — seed 표준편차(0.0006~0.0018)보다 작아 **통계적으로 유의미하지 않다.**
2. 오히려 τ=0.71의 seed 표준편차(0.0018)가 τ=0.7의 표준편차(0.0006)보다 **3배 크다** — peak 근방일수록 seed에 따라 결과가 더 민감하게 흔들린다는 뜻. 봉우리 꼭대기에 정확히 서려다 안정성을 잃는 것보다, 살짝 못 미치더라도 더 평평하고 안정적인 지점(τ=0.7)을 쓰는 게 안전하다.

**결론**: τ=0.70, actual 가중을 최종안으로 확정한다. (필요하면 나중에 `train.ipynb`에서 0.70~0.71 둘 다 놓고 seed를 더 늘려 재확인해도 되지만, 지금 단계에서는 차이가 노이즈 수준이라 우선순위가 낮다.)

### 9-8. 리더보드 1등과 비교 — Score를 1-NMAE/FICR로 분해

**결론 먼저**: 지금까지는 `metric()`이 반환하는 3개 값 중 합산 점수(`score`)만 봤다. 리더보드 1등 점수(2026-07-22 확인)와 비교하려면 성분별로 나눠 봐야 한다 — 1등: **Score 0.66961 = 0.5×(1-NMAE 0.89046) + 0.5×FICR 0.44876**.

우리 현재 최종안(τ=0.70, actual 가중, 기본 하이퍼파라미터)의 3-fold 평균을 같은 방식으로 분해해서 나란히 놓고, NMAE 쪽이 부족한지 FICR 쪽이 부족한지 확인한다. `FICR`은 오차율 구간(≤6%→4원, ≤8%→3원, 초과→0원)의 계단 함수라 **NMAE를 아주 조금만 줄여도 원 단위 구간을 넘어서면 FICR이 크게 뛸 수 있다** — 그래서 두 지표가 따로 움직인다.

In [ ]:
def train_and_score_fold_decomposed(
    fold_train, fold_valid, params, feature_cols=FEATURE_COLS, early_stop_frac=0.15,
    seed=SEED, quantile_alpha=None, use_sample_weight=False,
):
    features = feature_cols + ["group_id"]
    long_df = to_long_ext(fold_train, feature_cols)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    loss_function = f"Quantile:alpha={quantile_alpha}" if quantile_alpha is not None else "MAE"
    model = CatBoostRegressor(loss_function=loss_function, random_seed=seed, verbose=False, **params)

    fit_kwargs = {}
    if use_sample_weight:
        fit_kwargs["sample_weight"] = fit_rows["actual_kwh"].to_numpy()

    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"], early_stopping_rounds=100, verbose=False,
        **fit_kwargs,
    )

    pred = {}
    for g in TARGET_COLS:
        valid_g = fold_valid.copy()
        valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
        pred[g] = model.predict(valid_g[features]) * CAPACITY_KWH[g]

    answer_df = make_answer_df(fold_valid)
    pred_df = make_pred_df(fold_valid, pred)
    return metric(answer_df, pred_df)  # (score, one_minus_nmae, ficr)


decomposed_rows = []
for fold in FOLDS:
    fold_train, fold_valid = make_fold_frames(fold)
    score, one_minus_nmae, ficr = train_and_score_fold_decomposed(
        fold_train, fold_valid, DEFAULT_PARAMS, quantile_alpha=best_tau, use_sample_weight=True,
    )
    decomposed_rows.append({"fold": fold["name"], "Score": score, "1-NMAE": one_minus_nmae, "FICR": ficr})

decomposed_df = pd.DataFrame(decomposed_rows)
decomposed_mean = decomposed_df[["Score", "1-NMAE", "FICR"]].mean()

leaderboard_1st = pd.Series({"Score": 0.66993, "1-NMAE": 0.8808, "FICR": 0.45905})  # 2026-07-22 갱신 확인

compare_df = pd.DataFrame({
    "우리(tau=0.70, actual 가중, 3-fold 평균)": decomposed_mean,
    "리더보드 1등(2026-07-22 갱신)": leaderboard_1st,
})
compare_df["차이(1등-우리)"] = compare_df["리더보드 1등(2026-07-22 갱신)"] - compare_df["우리(tau=0.70, actual 가중, 3-fold 평균)"]

print(decomposed_df)
print()
compare_df

**해석**: 우리 3-fold 평균(τ=0.70, actual 가중) — Score 0.6219, 1-NMAE 0.8599, FICR 0.3839. 리더보드 1등(2026-07-22 갱신) — Score 0.6699, 1-NMAE 0.8808, FICR 0.4591. 차이는 **1-NMAE +0.021, FICR +0.075** — **FICR 격차가 NMAE 격차의 3.6배**다. 즉 우리가 따라잡아야 할 부분은 "오차를 평균적으로 더 줄이는 것"보다 "오차율 6%/8% 구간(계단 함수 경계)을 더 많은 시간대에서 넘기는 것"에 가깝다.

**참고 — 앞서 참고했던 1등 스냅샷과 지금 1등은 다른 사람이다**: 이전에 참고했던 스냅샷(Score 0.66961, 1-NMAE 0.89046, FICR 0.44876)과 지금 확인한 1등(Score 0.66993, 1-NMAE 0.8808, FICR 0.45905)은 **같은 사람이 전략을 바꾼 게 아니라 순위가 갈린 두 명의 서로 다른 결과**다. 그래서 "1등이 NMAE를 희생하고 FICR을 올리는 쪽으로 움직였다"는 식의 인과관계 해석은 근거가 없다 — 정정한다. 다만 두 스냅샷 모두 **비슷한 총점을 서로 다른 (1-NMAE, FICR) 조합으로 만들어낸다**는 사실 자체는 남는다: 상위권에는 "NMAE를 더 낮추는 전략"과 "FICR을 더 높이는 전략"이 둘 다 비슷한 총점에 도달할 수 있는 여지가 있다는 뜻으로, 우리에게 특정 방향(예: FICR 집중)이 유일한 정답이라고 단정할 근거는 아니다.

**결론적으로 확실한 것은**: 우리 결과와 지금 1등 사이의 격차 자체(FICR 격차가 NMAE 격차의 3.6배)뿐이다. 이것만으로 "FICR을 더 파야 한다"는 우선순위 판단은 유효하지만, 그 근거를 "1등의 전략 변화"에서 끌어오는 것은 잘못된 추론이었다.

**주의 — 이 비교는 어림값이다**: 우리 숫자는 2022~2024년 데이터로 만든 3-fold CV 평균이고, 리더보드 1등은 실제 2025년 테스트 데이터 예측 결과다. 기간·기상 조건이 다르므로 직접적인 "같은 시험 점수 비교"는 아니고, **어느 쪽 성분(NMAE vs FICR)이 상대적으로 더 벌어져 있는지 방향을 보는 용도**로만 쓴다. 위 셀을 실행해서 표를 확인한 뒤, 어느 성분 격차가 더 큰지에 따라 다음 우선순위(오차율 자체를 줄일지 vs 계단 구간을 넘기는 데 집중할지)를 정하자.

## 10. 오류 분석 — FICR 격차의 원인 찾기

**결론 먼저**: 9-8절에서 확인했듯 우리와 리더보드 1등의 격차는 FICR 쪽(0.075)이 1-NMAE 쪽(0.021)보다 3.6배 크다. `ensemble-final` 스킬 2절 체크리스트(풍속 구간별, 시간대/월별, 그룹별, **FICR 관점 — 오차율 6~8% 경계에 걸린 시간들의 공통점**)를 따라, 확정한 최종안(τ=0.70, actual 가중, 기본 하이퍼파라미터)의 검증 예측을 행 단위로 뜯어서 어떤 조건에서 6%/8% 경계를 못 넘는지 살펴본다.

**왜 앙상블보다 먼저 오류 분석인가**: `ensemble-final` 스킬 순서상 앙상블은 "분산을 줄여 극단 오차를 낮추는" 기법이라 일반적으로 도움이 되지만, 특정 조건(예: 특정 풍속 구간)에서 구조적으로 편향돼 있다면 그 원인을 먼저 알아야 한다 — 편향은 앙상블로 잘 안 없어지고(여러 모델이 같은 편향을 공유하기 쉬움), 원인 조건을 알면 그 구간만 피처 보강·구간별 보정 등으로 직접 대응할 수 있다.

**어떻게 오류 데이터를 만드는가**: 3개 fold 각각의 검증 구간(fold1=2023H2, fold2=2024H1, fold3=2024H2, 겹치지 않음)에 대해 그 fold의 학습 데이터로 학습한 모델의 예측을 모아 하나의 표로 합친다. 이렇게 하면 2023-07~2024-12 전체 기간을 "그 시점에 실제로 쓸 수 있었던 모델"의 예측으로 커버하는, OOF(out-of-fold — 학습에 안 쓰인 구간의 예측만 모은 것)와 비슷한 표가 된다. 다만 fold1/2/3은 서로 다른 학습 윈도우로 학습된 모델이라는 점은 감안한다(완전히 동일한 모델은 아님).

In [ ]:
def build_oof_error_df(params, quantile_alpha=None, use_sample_weight=False, feature_cols=FEATURE_COLS, seed=SEED):
    features = feature_cols + ["group_id"]
    loss_function = f"Quantile:alpha={quantile_alpha}" if quantile_alpha is not None else "MAE"
    rows = []

    for fold in FOLDS:
        fold_train, fold_valid = make_fold_frames(fold)
        long_df = to_long_ext(fold_train, feature_cols)
        long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
        long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
        n_early = int(len(long_sorted) * 0.15)
        fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

        model = CatBoostRegressor(loss_function=loss_function, random_seed=seed, verbose=False, **params)
        fit_kwargs = {"sample_weight": fit_rows["actual_kwh"].to_numpy()} if use_sample_weight else {}
        model.fit(
            fit_rows[features], fit_rows["utilization"],
            eval_set=(early_rows[features], early_rows["utilization"]),
            cat_features=["group_id"], early_stopping_rounds=100, verbose=False, **fit_kwargs,
        )

        for g in TARGET_COLS:
            valid_g = fold_valid.copy()
            valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
            pred_kwh = np.clip(model.predict(valid_g[features]) * CAPACITY_KWH[g], 0, CAPACITY_KWH[g])
            mask = valid_g[g].notna().to_numpy()
            rows.append(pd.DataFrame({
                "fold": fold["name"],
                "forecast_kst_dtm": valid_g.loc[mask, "forecast_kst_dtm"].values,
                "group": g,
                "actual": valid_g.loc[mask, g].values,
                "pred": pred_kwh[mask],
                "ws10m": valid_g.loc[mask, f"{g}_ldaps_ws10m"].values,
            }))

    return pd.concat(rows, ignore_index=True)


error_df = build_oof_error_df(DEFAULT_PARAMS, quantile_alpha=0.70, use_sample_weight=True)
error_df["capacity"] = error_df["group"].map(CAPACITY_KWH)
error_df["error_rate"] = (error_df["actual"] - error_df["pred"]).abs() / error_df["capacity"]
error_df["is_scored"] = error_df["actual"] >= 0.10 * error_df["capacity"]
error_df["price_krw"] = np.select(
    [error_df["error_rate"] <= 0.06, error_df["error_rate"] <= 0.08], [4, 3], default=0,
)
error_df["hour"] = error_df["forecast_kst_dtm"].dt.hour
error_df["month"] = error_df["forecast_kst_dtm"].dt.month

scored_df = error_df[error_df["is_scored"]].copy()
print(f"전체 {len(error_df)}행 중 채점 대상(actual>=10%용량) {len(scored_df)}행 ({len(scored_df) / len(error_df):.1%})")
scored_df["error_rate"].describe()

### 10-1. 그룹별 오차율·경계 통과율

그룹마다 터빈 기종·위치·데이터 기간이 달라(`wind-domain-features` 스킬 7절) 오차 특성이 다를 수 있다. 그룹별로 평균 오차율과 "6% 이내 비율"/"8% 이내 비율"을 본다. 특히 라벨 기간이 짧은 group_3(2023-01부터)이 다른 그룹보다 불안정한지 확인한다.

In [ ]:
group_summary = scored_df.groupby("group").agg(
    n=("error_rate", "size"),
    mean_error_rate=("error_rate", "mean"),
    pct_within_6pct=("error_rate", lambda s: (s <= 0.06).mean()),
    pct_within_8pct=("error_rate", lambda s: (s <= 0.08).mean()),
    mean_price_krw=("price_krw", "mean"),
)
group_summary

### 10-2. 풍속 구간별 오차율 — 파워커브 급경사 구간 확인

`wind-domain-features` 스킬 1절: 파워커브는 cut-in(~3 m/s)과 rated(~11~13 m/s) 근처에서 급경사(풍속이 조금만 달라도 발전량이 크게 바뀜) 구간이라 오차가 커지기 쉽다. 여기서 쓰는 `ws10m`은 허브高(80~100 m)가 아니라 지상 10 m 풍속이라 절대적인 cut-in/rated 수치와는 다르지만(윈드시어로 허브 높이가 더 빠름), **상대적인 저/중/고풍속 구간 비교**에는 쓸 수 있다. 그룹별 표본 수를 맞추기 위해 그룹 전체를 합쳐 5분위(quintile)로 나눈다.

In [ ]:
scored_df["ws_bin"] = pd.qcut(scored_df["ws10m"], 5)

ws_summary = scored_df.groupby("ws_bin", observed=True).agg(
    n=("error_rate", "size"),
    mean_ws10m=("ws10m", "mean"),
    mean_error_rate=("error_rate", "mean"),
    pct_within_6pct=("error_rate", lambda s: (s <= 0.06).mean()),
    pct_within_8pct=("error_rate", lambda s: (s <= 0.08).mean()),
)
ws_summary

### 10-3. 시간대별·월별 오차율

산악풍은 주야 순환(낮/밤 대류 차이)과 계절 패턴이 뚜렷하다(`wind-domain-features` 스킬 5절). 시간대(0~23시)와 월(1~12월)별로 평균 오차율과 6% 이내 비율을 본다.

In [ ]:
hour_summary = scored_df.groupby("hour").agg(
    n=("error_rate", "size"),
    mean_error_rate=("error_rate", "mean"),
    pct_within_6pct=("error_rate", lambda s: (s <= 0.06).mean()),
)

month_summary = scored_df.groupby("month").agg(
    n=("error_rate", "size"),
    mean_error_rate=("error_rate", "mean"),
    pct_within_6pct=("error_rate", lambda s: (s <= 0.06).mean()),
)

print("시간대별:")
print(hour_summary)
print("\n월별:")
print(month_summary)

### 10-4. 경계 근처(6%/8%) 시간대의 공통점 — FICR에 가장 직접적인 분석

`ensemble-final` 스킬 2절이 명시한 FICR 관점 체크: "오차율 6~8% 경계에 걸린 시간들의 공통점 (어떤 조건에서 아깝게 이탈하는가)". 오차율이 6%/8% 바로 위(±1%p)에 있는 시간대만 뽑아서, 그룹·풍속구간별로 어디에 몰려 있는지 본다. 여기 몰린 조건이 있다면 "그 구간만 오차를 1~2%p 더 줄이면 FICR이 즉시 오르는" 가장 효율 좋은 개선 지점이다.

In [ ]:
near_6pct = scored_df[(scored_df["error_rate"] > 0.05) & (scored_df["error_rate"] <= 0.07)]
near_8pct = scored_df[(scored_df["error_rate"] > 0.07) & (scored_df["error_rate"] <= 0.09)]
over_8pct = scored_df[scored_df["error_rate"] > 0.08]

print(f"6% 경계 근처(5~7%) 시간대: {len(near_6pct)}건 ({len(near_6pct) / len(scored_df):.1%})")
print(near_6pct.groupby("group").size())
print(near_6pct.groupby("ws_bin", observed=True).size())

print(f"\n8% 경계 근처(7~9%) 시간대: {len(near_8pct)}건 ({len(near_8pct) / len(scored_df):.1%})")
print(near_8pct.groupby("group").size())
print(near_8pct.groupby("ws_bin", observed=True).size())

print(f"\n8% 초과(가격 0원 구간): {len(over_8pct)}건 ({len(over_8pct) / len(scored_df):.1%})")
print(over_8pct.groupby("group").size())
print(over_8pct.groupby("ws_bin", observed=True).size())

### 10-5. 종합 해석

**가장 먼저 봐야 할 숫자**: 채점 대상 22,160시간 중 **60.5%(13,402시간)가 오차율 8% 초과 — 가격이 0원**이다. 반면 6%/8% 경계 바로 근처(±1%p)에 있는 "아깝게 놓친" 시간대는 각각 9.4%/8.8%뿐이다. 즉 **FICR이 낮은 주된 이유는 "경계를 살짝 못 넘는 것"이 아니라 "애초에 오차가 크게 나는 시간이 절반 넘게 있는 것"**이다 — τ를 미세 조정하는 식의 보정(9절에서 이미 거의 다 짜냄)으로는 이 부분을 더 줄이기 어렵고, 예보 자체의 정확도(풍속 예보 오차, 그리고 그것이 v³로 증폭되는 구조)를 개선해야 하는 문제로 보인다.

**그룹별 — group_3이 뚜렷하게 약하다**: 8% 이내 비율이 group_1(42.3%)·group_2(41.7%) 대비 group_3은 34.0%로 낮고, **8% 초과(0원) 비율도 group_3이 66.0%로 group_1(57.7%)·group_2(58.3%)보다 눈에 띄게 높다.** `wind-domain-features` 스킬 7절 근거대로 group_3은 라벨 기간이 2023-01부터로 짧아(다른 그룹보다 학습에 쓸 수 있는 과거 데이터가 약 1년 적음) 그룹별 특성(파워커브, 최근접 격자)을 모델이 덜 배웠을 가능성이 높다.

**풍속 구간별 — 파워커브 이론과 정확히 들어맞는다**: 오차율이 가장 낮은 구간은 **가장 낮은 풍속 구간**(10m 풍속 0.1~4.2 m/s, 오차율 12.1%, 8%이내 46.6%)이고, 오차율이 가장 높은 구간은 **중간 풍속 구간**(10m 풍속 6.6~8.3 m/s, 오차율 16.1%, 8%이내 33.8%)이다. 가장 높은 풍속 구간(8.3~17.4 m/s)에서는 오차율이 다시 13.7%로 낮아진다. 이는 `wind-domain-features` 스킬 1절의 파워커브 이론과 정확히 일치한다 — 저풍속(컷인 근처)은 출력이 0에 가까워 맞히기 쉽고, 정격 근처로 올라가는 급경사 구간(우리 데이터의 10m 풍속으로는 중간 구간에 해당 — 윈드시어를 감안하면 허브高에서는 이 구간이 실제 정격 근접 풍속대일 가능성이 높다)은 풍속 오차가 발전량 오차로 크게 증폭되고, 정격 근처·이상(고풍속 구간)은 출력이 포화돼 있어 풍속 오차의 영향이 줄어든다.

**시간대·월별 — 부차적 신호**: 시간대는 큰 차이가 없지만 이른 아침(6~8시)·저녁(20~23시) 시간대가 조금 더 나쁘고 정오~오후 초반이 조금 낫다(대류 순환의 영향으로 추정). 월별로는 9월(오차율 10.8%, 6%이내 40.2%)이 가장 좋고 1월(16.0%, 27.7%)·3월(15.1%, 26.2%)·7월(15.3%, 25.3%)이 나쁘다 — 겨울철 변동성 큰 바람과 장마·태풍철(7월) 불안정성이 원인으로 추정되나, 뚜렷한 단조 패턴은 아니라 그룹·풍속 대비 우선순위는 낮다.

**다음 단계 우선순위 제안**:
1. **group_3 전용 대응**: 그룹별로 별도 τ 튜닝, 또는 group_3에 한해 표본 가중치를 더 주는 방식 검토
2. **중간 풍속 구간(급경사 구간) 정확도 개선**: 이 구간에 특화된 피처(풍속 변화율, gust 비율 등 이미 있는 피처들의 중요도를 이 구간에서만 따로 확인) 또는 구간별 보정항 검토
3. τ 미세조정으로 짜낼 수 있는 여지는 이미 9절에서 거의 소진됐다고 보고, 이 오류 분석 결과를 바탕으로 피처·구간별 접근에 시간을 쓰는 게 `model-tuning` 스킬 4절("튜닝으로 짜낸 0.001보다 피처 하나의 0.01이 크다")과 일치

## 요약

- `model-tuning`/`timeseries-validation` 스킬 기준, 확장 윈도우 3-fold CV로 바꿔 단일 fold 과적합 위험을 줄였다
- **8절 하이퍼파라미터 튜닝(Optuna 30 trial)**: 개선폭 +0.000045 — seed 표준편차보다 작아 "효과 미미"로 정직하게 기록
- **9절 손실 함수·가중치 조정**: actual 가중만으로는 미미(+0.0004)했지만, **분위수 회귀 τ=0.70 + actual 가중으로 +0.0305 개선**(seed 표준편차의 47.6배 — 안정적) — 이 노트북의 핵심 발견
- τ를 0.70~0.72까지 세밀 탐색해 진짜 peak(τ=0.71)를 확인했지만, 개선폭이 노이즈 수준이고 seed 표준편차가 3배 커서 **τ=0.70을 최종안으로 확정**
- 리더보드 1등(2026-07-22)과 Score를 1-NMAE/FICR로 분해해 비교: **FICR 격차(0.075)가 NMAE 격차(0.021)의 3.6배** — 다음 개선 우선순위는 평균 오차 축소보다 오차율 6%/8% 계단 경계를 넘기는 쪽
- **10절 오류 분석**: 채점 대상의 **60.5%가 오차율 8% 초과(0원)** — 경계 근처(±1%p)는 9% 남짓뿐이라 "미세 보정"보다 "예보 오차 자체 축소"가 문제. **group_3이 뚜렷하게 약함**(8% 초과 비율 66.0% vs group_1/2 57~58%), **중간 풍속 구간에서 오차 최대**(파워커브 급경사 구간 이론과 일치)

**최종 확정안**: 통합모델(CatBoost) + 기본 하이퍼파라미터(`DEFAULT_PARAMS`) + 분위수 회귀(τ=0.70) + actual 가중 학습. 8절에서 찾은 `BEST_PARAMS`(하이퍼파라미터 튜닝 결과)는 효과가 노이즈 수준이라 채택하지 않는다.

**다음 할 일**:
1. `train.ipynb`로 넘어가 τ=0.70 + actual 가중 설정을 하드코딩해 전체 데이터로 재학습 (`ensemble-final` 스킬 4~5절)
2. group_3 전용 대응(별도 τ 또는 표본 가중치)과 중간 풍속 구간 피처 보강은 시간 여유에 따라 재학습 이후 별도 실험으로 고려
3. `HANDOFF.md` 갱신 — 세션 종료 시 `session-handoff` 스킬 참조

## 11. group_3 전용 대응 실험

10절 오류 분석에서 확인된 group_3 약점(채점 대상 중 오차율 8% 초과 비율 66.0% — group_1/2의 57~58%보다 뚜렷이 나쁨)에 대응한다. 민석님이 다음 우선순위로 이 실험을 선택했다.

두 가지 접근을 각각 3-fold CV로 검증한다.

1. **group_3 표본 가중치 강화**: 통합모델 구조는 그대로 두고, 학습 시 group_3 행에 추가 가중치 배수를 곱한다(`leakage-guard` 관점: 이미 학습에 쓰던 행의 가중치만 바꾸는 것이라 새 정보 유입이 없고 누수 위험 없음).
2. **group_3 전용 τ**: 그룹별로 완전히 별도 모델을 쓰면 `04_model_selection`에서 확인한 "통합모델이 그룹별모델보다 낫다"(풀링 효과, 0.5971 vs 0.5868)를 잃는다. 그래서 **풀링 구조(전 그룹 데이터로 학습)는 유지**하면서, τ가 다른 모델을 여러 개 학습해두고 **group_3 예측만 다른 τ 모델 것으로 교체**하는 방식으로 절충한다. group_1/2는 기존 최적 모델(τ=`best_tau`) 그대로 사용.

각 실험에서 그룹별 NMAE/FICR을 분해해, "group_3만 좋아지고 전체는 나빠지는" 트레이드오프가 있는지 함께 확인한다.

*(이 절은 실패/미채택으로 결론나 코드는 삭제하고 마크다운 기록만 남겼다 — 실행 결과는 각 소절 설명과 마지막 종합 해석, `reports/05_tuning.md`를 참고.)*

### 11-1. 그룹별 지표 분해 헬퍼 함수

`src/metric.py`는 수정하지 않는다. 대신 이 노트북 안에서만, `metric()`이 3개 그룹을 평균 내기 **전 단계**(그룹별 NMAE·FICR)까지 반환하도록 계산 로직을 그대로 복제한 헬퍼를 만든다. 그래야 "group_1/2는 기존 예측, group_3만 다른 모델 예측"처럼 섞어서 전체 Score를 재구성할 수 있다.

마지막 셀에서 이 분해 함수로 재조합한 Score가 기존 `metric()` 결과와 정확히 같은지 검산한다(`notebook-workflow` 스킬 — 중간 결과 확인 원칙).

### 11-2. 그룹별 예측을 섞을 수 있는 학습/예측 헬퍼 함수

기존 `train_and_score_fold_ext`는 학습·예측·채점을 한 번에 처리하지만, 이번엔 "모델 A로 group_1/2를 예측하고 모델 B로 group_3만 예측"처럼 섞어야 하므로 학습과 예측을 분리한다. `group_weight_multiplier`는 11-3(표본 가중치 실험)에서 사용한다.

In [10]:
def train_fold_model(
    fold_train, params, feature_cols=FEATURE_COLS, early_stop_frac=0.15, seed=SEED,
    quantile_alpha=None, use_sample_weight=False, group_weight_multiplier=None,
):
    features = feature_cols + ["group_id"]
    long_df = to_long_ext(fold_train, feature_cols)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    loss_function = f"Quantile:alpha={quantile_alpha}" if quantile_alpha is not None else "MAE"
    model = CatBoostRegressor(loss_function=loss_function, random_seed=seed, verbose=False, **params)

    weight = fit_rows["actual_kwh"].to_numpy(dtype=float) if use_sample_weight else None
    if group_weight_multiplier:
        base = weight if weight is not None else np.ones(len(fit_rows))
        mult = fit_rows["group_id"].astype(int).map(group_weight_multiplier).fillna(1.0).to_numpy(dtype=float)
        weight = base * mult

    fit_kwargs = {"sample_weight": weight} if weight is not None else {}
    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"], early_stopping_rounds=100, verbose=False,
        **fit_kwargs,
    )
    return model


def predict_group(model, fold_valid, g, feature_cols=FEATURE_COLS):
    features = feature_cols + ["group_id"]
    valid_g = fold_valid.copy()
    valid_g["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(valid_g), categories=GROUP_ID_CATEGORIES)
    return model.predict(valid_g[features]) * CAPACITY_KWH[g]

### 11-3. group_3 표본 가중치 배수 실험

기존 actual 가중(발전량이 클수록 가중치가 큼) 위에, group_3 행 전체에 추가 배수를 곱한다. 배수를 키울수록 모델이 group_3 오차를 줄이는 데 더 신경 쓰지만, 풀링된 단일 모델이므로 group_1/2가 희생될 위험이 있다 — 그래서 매 배수마다 그룹별 NMAE/FICR을 함께 확인한다.

### 11-4. group_3 전용 τ 실험 (풀링 구조 유지)

τ 후보마다 전체 데이터(3개 그룹 풀링)로 모델을 새로 학습한 뒤, **group_1/2 예측은 baseline(τ=`best_tau`) 모델 것을 그대로 쓰고 group_3 예측만 후보 τ 모델로 교체**해서 전체 Score를 재계산한다. 이렇게 하면 "풀링 효과는 유지하면서 group_3에만 다른 분위수를 적용"한 순수 효과를 볼 수 있다.

### 11-5. 유망한 방향 seed 안정성 재검증

11-3(표본 가중치)과 11-4(전용 τ) 중 CV 평균이 baseline보다 더 개선된 쪽을 골라 seed 3개(42/7/2024)로 재검증한다(`model-tuning` 스킬 3절 — 단일 seed 개선은 신뢰하지 않는다). 둘 다 baseline을 못 넘으면 "group_3 전용 대응으로는 개선을 찾지 못했다"고 정직하게 기록한다. 비교 기준은 9-3(`tau_seed_mean`, τ=0.70·actual 가중만 적용한 seed 평균)이다.

### 11-6. 종합 해석

**검산(11-1)**: 그룹별 분해 함수(`compute_group_metrics`/`combined_score`)로 재조합한 Score(0.603306)가 `metric()` 직접 계산과 소수점 6자리까지 정확히 일치 — 이후 실험 결과를 신뢰할 수 있다.

**11-3, 표본 가중치**: baseline(w3=1.0, 0.6219) 대비 최고 배수(w3=2.0)에서 0.6225로 **+0.0006** 개선. w3를 더 올릴수록(3.0, 5.0) 오히려 다시 떨어져 U자 형태 — 과도한 가중치는 group_1/2를 희생시키는 역효과가 실제로 나타난다.

**11-4, 전용 τ**: baseline(τ3=0.70, 0.6219) 대비 최고(τ3=0.75)에서 0.6224로 **+0.0005** 개선. group_3 NMAE는 τ3가 오를수록 계속 악화되는데(0.60→0.85 구간 0.145→0.192로 단조 증가) FICR은 0.75까지 오르다가 0.80부터 꺾인다 — τ를 높일수록 "값을 더 크게 예측해 상한 경계를 넘기려는" 시도가 FICR은 잠깐 개선시키지만 그 대가로 평균 오차(NMAE)를 계속 깎아먹는 전형적인 trade-off 구조.

**11-5, seed 재검증(핵심)**: 두 실험 중 더 나았던 표본 가중치(w3=2.0)를 seed 3개(42/7/2024)로 재검증하니 평균 0.6213, 표준편차 0.0013 — **baseline(τ=0.70, actual 가중, seed 평균 0.6213)과 완전히 동일하다(개선폭 0.0000, 표준편차의 0배)**. 단일 seed에서 보였던 +0.0006은 순수한 seed 노이즈였음이 확인됐다.

**결론**: group_3 전용 대응(표본 가중치든 전용 τ든)으로는 유의미한 개선을 찾지 못했다. 이는 10절 오류 분석의 핵심 발견("채점 대상의 60.5%가 오차율 8% 초과 — FICR이 낮은 원인은 경계 미세 이탈이 아니라 예보 자체의 오차가 크게 나는 시간이 절반 이상이라는 것")과 정확히 일치하는 결과다. group_3의 약점은 손실 함수·가중치 같은 "학습 방식 조정"으로 고칠 수 있는 종류가 아니라, 예보 정확도 자체(더 나은 피처, 혹은 group_3 학습 데이터 부족 자체)를 개선해야 하는 문제로 보인다.

**다음 단계**: group_3 전용 대응은 이 실험으로 탐색을 마무리하고(τ=0.70 + actual 가중, 표본 가중치 없음 — 기존 확정안 유지), HANDOFF `다음 할 일` 2번(중간 풍속 구간 6.6~8.3 m/s 특화 피처 보강)으로 우선순위를 옮긴다.

## 12. 중간 풍속 구간 피처 보강 — 풍향×풍속 상호작용 + shear_alpha 재검토

10절 오류 분석에서 중간 풍속 구간(10m 풍속 6.6~8.3 m/s)이 파워커브 급경사 구간이라 오차가 가장 크다는 걸 확인했고, 11절(group_3 전용 τ/가중치)로는 개선을 찾지 못했다. 이번엔 손실 함수·가중치가 아니라 **피처 자체**를 보강해본다(`model-tuning` 스킬 4절: "튜닝으로 짜낸 0.001보다 피처 하나의 0.01이 크다"). `wind-domain-features` 스킬 기준으로 비용이 낮은 두 후보를 고른다.

1. **풍향×풍속 상호작용**: 산악 지형(태백 가덕산)은 풍향에 따라 지형 가속·차폐가 달라진다(`wind-domain-features` 스킬 2절) — 지금은 풍속·풍향(sin/cos)이 따로만 있어 GBDT가 상호작용을 스스로 찾아야 한다. 그룹별 LDAPS(`ws10m × wd10m_sin/cos`)와 공유 GFS(`ws100m × wd100m_sin/cos`)에 각각 곱셈 항을 추가한다.
2. **`gfs_shear_alpha`(윈드시어 지수) 재검토**: `04_model_selection`에서 ablation 손해가 -0.0003으로 작다는 이유로 제거했지만(v1 parquet엔 여전히 남아있음), 그건 quantile 회귀 도입 **이전** 구모델·**전체** CV 기준이었다. 최종 확정 모델(τ=0.70+actual 가중)로, 그리고 문제가 된 중간 풍속 구간 오차에 한정해서도 다시 확인해본다.

**leakage 체크**(`leakage-guard` 스킬): 둘 다 예측기준시점에 이미 공개된 예보값(풍속·풍향·시어)의 결정론적 변환(곱셈, 로그비)이다. 새로운 미래 정보가 들어오지 않으므로 안전.

*(이 절은 실패/미채택으로 결론나 코드는 삭제하고 마크다운 기록만 남겼다 — 실행 결과는 각 소절 설명과 마지막 종합 해석, `reports/05_tuning.md`를 참고.)*

### 12-1. 피처 준비

v1 parquet에서 `gfs_shear_alpha`만 시각 기준으로 병합하고, 기존 v2 컬럼으로 상호작용 피처를 계산한다. parquet 파일은 새로 저장하지 않는다 — 이번엔 후보 피처가 실제로 도움이 되는지 확인하는 실험 단계이고, 채택되면 그때 `03_features`/`04_model_selection` 쪽에 정식 반영한다.

### 12-2. 4가지 피처셋 조합 CV 비교

baseline(기존 50개) / +상호작용 / +shear_alpha / +둘 다, 4가지를 같은 3-fold CV·같은 최종 설정(`DEFAULT_PARAMS`, τ=`best_tau`, actual 가중)으로 비교한다. `train_features_ext`를 쓰도록 `make_fold_frames`를 이 셀 안에서만 다시 정의한다(기존 `make_fold_frames`는 `train_features`를 참조해서 그대로는 확장 컬럼을 못 봄).

### 12-3. 최선 조합 seed 안정성 재검증

11절 교훈(단일 CV 개선이 seed 노이즈였던 경험)을 그대로 적용한다. 최고 조합이 baseline을 못 이기면 여기서 "개선 없음"으로 정직하게 마무리한다.

### 12-4. 중간 풍속 구간(6.6~8.3 m/s) 오차 재확인

전체 Score는 3개 그룹·전체 기간 평균이라 중간 풍속 구간만의 효과가 묻힐 수 있다. 승자 피처셋(seed 재검증까지 통과했을 때만 의미 있음)으로 10-2와 같은 방식의 OOF 오류표를 baseline·승자 각각 이 자리에서 새로 만들어 비교한다(10절 `scored_df`를 재사용하지 않음 — 12절만 따로 실행할 수 있도록 self-contained로 설계, 0~4절+9-2+9-3만 있으면 된다).

### 12-5. 종합 해석

**12-2, 4개 피처셋 비교**: baseline(50개, 0.6219)이 **모든 확장 조합보다 높거나 같았다** — +상호작용(58개, 0.6200, -0.0019), +shear_alpha(51개, 0.6203, -0.0016), +둘다(59개, 0.6218, -0.0000). 어느 조합도 baseline을 이기지 못해 12-3의 seed 재검증까지 갈 필요조차 없었다(12-3이 자동으로 이를 감지하고 건너뜀).

**12-4, 구간별 재확인**: 이긴 조합이 없어 `winning_feature_cols = FEATURE_COLS`(baseline)로 처리됐고, 그래서 delta가 전 구간에서 정확히 0으로 나온다 — 버그가 아니라 "새로 시도할 피처셋이 없다"는 걸 그대로 보여주는 정상 동작이다.

**왜 실패했는가(도메인 해석)**:
- **풍향×풍속 상호작용**: `sin(wd)`, `cos(wd)`, `ws`가 이미 개별 피처로 있어 CatBoost(대칭 트리)가 순차 분기로 상호작용을 어느 정도 스스로 찾을 수 있다. 곱셈항을 명시적으로 추가하면 기존 피처와 강하게 공선인 컬럼이 늘어나기만 해서(26k행이라는 제한된 데이터에서) 분할 품질을 오히려 흐릴 수 있다.
- **`gfs_shear_alpha` 재도입**: `04_model_selection`에서 이미 "ablation 손해 -0.0003(거의 없음) + 10/80/100m 원시 풍속이 있어 모델이 시어를 스스로 유추 가능 + 야간 무풍 구간에서 불안정"이라는 근거로 제거했던 결정이, quantile 회귀(τ=0.70)+actual 가중을 적용한 최종 모델에서도 **그대로 유효함이 재확인됐다** — 오히려 이번엔 -0.0016으로 더 뚜렷하게 손해였다.

**결론**: 저비용 피처 후보 2가지 모두 실패. `wind-domain-features`/`model-tuning` 스킬대로 도메인 근거는 있었지만, "모델이 이미 알고 있는 정보의 재포장"이라 새 정보가 되지 못했다. 남은 후보는 더 비용이 큰 방향(LDAPS 그룹별 gust_ratio 등 원시데이터 재처리)뿐이다.

**다음 단계 제안**: 11절(group_3 대응)·12절(피처 보강) 둘 다 실패한 건, 채점 대상의 60.5%가 오차율 8% 초과라는 10절 결론대로 **문제의 핵심이 손실함수·가중치·저비용 피처가 아니라 NWP 예보 자체의 정확도 한계**일 가능성을 계속 시사한다. 이 지점에서 (a) 더 비용이 큰 피처(LDAPS gust_ratio 등)를 시도할지, (b) 여기서 개선 탐색을 마무리하고 현재 확정 모델로 최종 제출 준비에 집중할지는 민석님과 논의해서 결정한다.

## 13. 미사용 원시변수 — LDAPS 5m 경계층 바람(XBLWS/YBLWS)

11절(group_3 대응)·12절(풍향×풍속 상호작용, shear_alpha)이 모두 실패했다. 두 실패의 공통점은 **이미 모델이 갖고 있는 정보를 다른 형태로 재포장**한 것뿐이었다는 점이다(12절 So-what). 이번엔 완전히 새로운 원시변수를 찾아본다.

`data_description.md`의 LDAPS 변수 목록을 다시 훑어보니, `heightAboveGround_5_XBLWS`/`YBLWS`(지상 5m 경계층 바람 X/Y 성분)가 지금까지 어떤 피처에도 쓰인 적이 없다. 원시 데이터로 직접 확인한 결과:

- 크기는 평균 ~0.15 m/s로 10m 풍속(~5.6 m/s)보다 훨씬 작은, **근접 지표층의 별도 성분**
- 10m 풍속과 상관 0.73 — 관련은 있지만 **완전 중복은 아님**(기존 50m 돌풍범위 피처와 고도·성격이 다름)

이걸로 그룹별 두 후보를 만든다:
- `{group}_ldaps_blws_mag` = 그룹가중 후 `sqrt(X²+Y²)` (크기 자체)
- `{group}_ldaps_blws_ratio` = `blws_mag / ws10m`(정규화, 근접 지표층 난류 대리변수 — `gfs_gust_ratio`와 같은 방식)

**leakage 체크**(`leakage-guard` 스킬): 예측기준시점에 이미 공개된 예보값의 그룹가중 변환일 뿐 — 안전. **품질 체크**: test에 1개 발표분(2025-07-18 06:00 근방, `test LDAPS 결측 3시각` 문제와 같은 케이스)에서 16개 결측이 있어 `03_features` A절과 동일한 방식(같은 발표분 내 시간보간)으로 먼저 채운다.

이 절은 03_features.ipynb를 아직 건드리지 않고 05_tuning 안에서만 실험한다(11·12절과 같은 원칙) — 개선이 확인되면 그때 `03_features.ipynb`에 정식 그룹으로 추가한다.

*(이 절은 실패/미채택으로 결론나 코드는 삭제하고 마크다운 기록만 남겼다 — 실행 결과는 각 소절 설명과 마지막 종합 해석, `reports/05_tuning.md`를 참고.)*

### 13-1. 피처 준비

`01_preprocessing` 산출물(`train_base_wide.parquet`/`test_base_wide.parquet`)에서 XBLWS/YBLWS 원시 컬럼을 그룹가중해 `blws_mag`를 만들고, v2 피처의 `{group}_ldaps_ws10m`으로 나눠 `blws_ratio`를 만든다. `GROUP_GRID_WEIGHTS`/`group_weighted_uv`는 `03_features.ipynb`와 정확히 같은 값·로직을 이 노트북에서 다시 정의한다(그 노트북을 실행하지 않고도 이 절만 돌아가게 하기 위함).

### 13-2. 4가지 피처셋 조합 CV 비교

baseline(기존 50개) / +blws_mag(53개) / +blws_ratio(53개) / +둘 다(56개)를 같은 3-fold CV·최종 설정(τ=`best_tau`, actual 가중)으로 비교한다. 이 절은 11·12절과 독립적으로 동작하도록 `make_fold_frames_blws`를 새로 정의한다(0~4절 + 9-2/9-3만 있으면 이 절이 단독으로 돌아간다).

### 13-3. 최선 조합 seed 안정성 재검증

11·12절과 같은 원칙 — baseline을 못 이기면 여기서 "개선 없음"으로 마무리한다.

### 13-4. 중간 풍속 구간(6.6~8.3 m/s) 오차 재확인

12-4와 같은 방식(self-contained) — baseline/승자 OOF를 이 자리에서 직접 계산해 문제의 구간에서 오차율이 줄었는지 확인한다.

### 13-5. 종합 해석

**13-2, 4개 피처셋 비교**: baseline(50개, 0.6219)이 세 번째로 다시 최고였다 — +blws_mag(53개, 0.6210, -0.0009), +blws_ratio(53개, 0.6204, -0.0016), +둘다(56개, 0.6211, -0.0008). 세 조합 모두 baseline보다 낮아 12절과 마찬가지로 seed 재검증(13-3)까지 갈 필요 없이 종료됐다. 13-4도 승자가 없어 delta=0(정상 동작).

**왜 실패했는가**: `blws_mag`가 10m 풍속과 상관 0.73으로 어느 정도 독립적인 신호였는데도 실패했다는 게 11·12절과는 조금 다른 지점이다. 두 가지로 해석할 수 있다.
1. 상관 0.73이라도 CatBoost 입장에서는 이미 있는 `ws10m`·`gust_range_50m`·`blh` 조합으로 비슷한 정보를 충분히 근사하고 있었을 가능성 — 즉 "새로운 원시변수"였지만 "새로운 예측정보"는 아니었을 수 있다.
2. `blws_mag` 자체의 절대 크기(~0.15 m/s)가 매우 작고 노이즈 대비 신호비가 낮아, 26k 행 규모에서 트리 분할에 안정적으로 기여하기엔 부족했을 수 있다(과적합 방지 관점에서 오히려 분할 후보만 늘려 손해).

**세 번의 시도(11·12·13절) 종합**:

| 절 | 시도 | 결과 |
|---|---|---|
| 11 | group_3 표본가중치·전용 τ | 실패 (seed 재검증에서 개선폭 0) |
| 12 | 풍향×풍속 상호작용, shear_alpha 재도입 | 실패 (모든 조합이 baseline보다 손해) |
| 13 | LDAPS 5m 경계층바람(blws_mag/ratio) | 실패 (모든 조합이 baseline보다 손해) |

손실함수·가중치 조정(11절), 저비용 파생 피처(12절), 완전히 새로운 원시변수(13절) 세 가지 서로 다른 성격의 레버를 모두 시도했고 전부 baseline을 못 넘었다. 이는 우연이 아니라 10절 오류분석의 진단("채점 대상의 60.5%가 오차율 8% 초과 — 문제는 경계 미세조정이 아니라 예보 자체의 오차")이 옳았다는 걸 세 번 독립적으로 재확인한 결과로 봐야 한다.

**결론**: 현재 피처셋(50개)·모델(CatBoost + τ=0.70 + actual 가중)이 이 데이터로 도달 가능한 성능에 가깝다고 판단한다. 추가 개선은 NWP 예보 자체의 정확도(우리가 통제할 수 없는 영역)에 달려있을 가능성이 높다. **여기서 개선 탐색을 마무리하고 최종 제출 준비(모델·설정 변경 없음, `train.ipynb`/`inference.ipynb` 그대로 유지)로 넘어가는 것을 제안한다.**

## 14. MLP + 미분 가능한 FICR 손실 (CatBoost와 블렌드)

11~13절은 전부 실패했다. 세 실패의 공통점은 CatBoost의 quantile/MAE 손실이라는 **간접적인 우회**였다는 점이다 — τ를 옮기거나(11절), 피처를 늘리거나(12·13절) 해도 결국 "평균 오차를 줄인다"는 목표 자체는 그대로였다.

민석님이 공유해주신 별도 파이프라인 기록(phase6_nn_metric_loss.md)을 보면, 같은 정체를 완전히 다른 방식으로 뚫었다: **채점 산식의 계단 함수(FICR: 오차율 6%/8% 경계에서 단가가 4→3→0원으로 뚝뚝 떨어짐)를 시그모이드로 미분 가능하게 근사해, 신경망으로 그 근사 손실을 직접 최적화**했다. 그 파이프라인에서 로컬 검증 +0.0151, 실제 리더보드에서도 +0.0160으로 재현됐다(CV 착시가 아님).

**이건 "신경망이 GBDT보다 낫다"는 이야기가 아니다.** CatBoost/LightGBM은 목적함수를 바꾸려면 1차 미분(gradient)뿐 아니라 2차 미분(hessian) 근사가 필요한데, 이 근사 없이 커스텀 손실을 넣으면 학습이 무너진다(그 문서 §2-2에서 실제로 CV 0.5641까지 폭락한 사례가 있다). 신경망은 backprop이라 이 제약이 없어서, **채점 산식을 그대로 손실로 쓸 수 있다는 게 핵심**이다.

이 절에서 하는 일:
1. `src/nn.py`에 MLP + 미분 가능한 FICR 손실(`soft_metric_loss`) 구현 (완료, 이 노트북 밖의 파일)
2. 우리 v2 피처셋(50개) + 3-fold CV로 MLP 단독 성능 확인
3. `T_soft`(계단을 얼마나 부드럽게 근사할지) 스윕, seed 안정성 확인
4. **CatBoost(11~13절 최종 확정 모델)와 블렌드** — 그룹별 블렌드 가중치까지 탐색

`torch`가 venv에 없으면 `pip install torch`(CPU 버전이면 충분, GPU 불필요)로 설치해야 한다. 학습 자체가 여러 판(T_soft 스윕 5개 × 3fold, seed 3개 × 3fold, 블렌드용 재학습 3fold)이라 이전 절들보다 오래 걸릴 수 있다.

### 14-1. 셋업

`torch`를 불러오고 `src/nn.py`를 import한다. `torch.nn`과 이름이 겹치지 않도록 `mlp_nn`이라는 별칭을 쓴다.

In [22]:
import torch

from src import nn as mlp_nn

print("torch version:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available(), "(CPU만 써도 충분 — 재현성 목적으로 일부러 CPU 고정)")

torch version: 2.13.0+cpu
CUDA 사용 가능: False (CPU만 써도 충분 — 재현성 목적으로 일부러 CPU 고정)


### 14-2. 피처 행렬 헬퍼

`group_id`를 원-핫(3열)으로 바꾸고, 수치 피처는 학습(fit) 파트로만 표준화(`mu`/`sd`)한 뒤 검증·테스트에 똑같이 적용한다(`leakage-guard` 원칙).

**결측 처리**: `*_diff_prev`류 피처(발표분 첫 시각엔 이전 시각이 없어 NaN, 하루 1096개)가 있다. CatBoost는 결측을 알아서 최적 분기 방향으로 처리하지만(그래서 지금까지 문제가 안 됐다), MLP 입력에 NaN이 들어가면 표준화·행렬곱 전체가 NaN으로 오염된다. "이전 시각 대비 변화를 알 수 없음"을 중립값 0(변화 없음)으로 채운다 — 미래 정보를 쓰는 게 아니라 그 시각에 이미 없는 정보를 중립값으로 대체하는 것이라 `leakage-guard` 위반이 아니다.

In [20]:
def build_mlp_features(df, feature_cols, mu=None, sd=None):
    group_onehot = pd.get_dummies(df["group_id"].astype(int), prefix="grp").reindex(
        columns=[f"grp_{i}" for i in range(3)], fill_value=0
    ).to_numpy(dtype=np.float32)

    # *_diff_prev류 피처는 발표분 첫 시각에 "이전 시각"이 없어 NaN이다(하루 1096개).
    # CatBoost는 결측을 스스로 처리하지만 MLP는 NaN이 그대로 전파되어 망가진다.
    # "이전 대비 변화를 알 수 없음"을 중립값 0(=변화 없음으로 간주)으로 채운다.
    num_x = df[feature_cols].fillna(0.0).to_numpy(dtype=np.float64)
    if mu is None:
        mu, sd = mlp_nn.fit_standardizer(num_x)
    num_x = mlp_nn.apply_standardizer(num_x, mu, sd).astype(np.float32)

    x = np.concatenate([num_x, group_onehot], axis=1)
    return x, mu, sd

### 14-3. 학습·예측·CV 함수

CatBoost 쪽(`train_fold_model`/`predict_group`, 11절)과 대응되는 MLP 버전. `to_long_ext`(9절)로 그룹 풀링 후 시간순 마지막 15%를 조기 종료용으로 떼어내는 방식은 CatBoost와 동일하게 맞춘다.

In [17]:
def train_mlp_fold(
    fold_train, feature_cols, seed=SEED, T_soft=0.006,
    hidden=(256, 256), dropout=0.15, early_stop_frac=0.15, verbose=False,
):
    long_df = to_long_ext(fold_train, feature_cols)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * early_stop_frac)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    fit_X, mu, sd = build_mlp_features(fit_rows, feature_cols)
    early_X, _, _ = build_mlp_features(early_rows, feature_cols, mu=mu, sd=sd)

    fit_util = fit_rows["utilization"].to_numpy(dtype=np.float32)
    fit_kwh = fit_rows["actual_kwh"].to_numpy(dtype=np.float32)
    fit_scored = fit_util >= 0.10

    early_util = early_rows["utilization"].to_numpy(dtype=np.float32)
    early_kwh = early_rows["actual_kwh"].to_numpy(dtype=np.float32)
    early_scored = early_util >= 0.10

    model, best_val_score, best_epoch = mlp_nn.train_mlp(
        fit_X, fit_util, fit_kwh, fit_scored,
        early_X, early_util, early_kwh, early_scored,
        input_dim=fit_X.shape[1], seed=seed, T_soft=T_soft,
        hidden=hidden, dropout=dropout, verbose=verbose,
    )
    return model, mu, sd, best_epoch


def predict_group_mlp(model, fold_valid, g, feature_cols, mu, sd):
    valid_g = fold_valid.copy()
    valid_g["group_id"] = GROUP_ID_MAP[g]
    x, _, _ = build_mlp_features(valid_g, feature_cols, mu=mu, sd=sd)
    pred_util = mlp_nn.predict_mlp(model, x)
    return pred_util * CAPACITY_KWH[g]


def cv_score_mlp(feature_cols=FEATURE_COLS, seed=SEED, T_soft=0.006, hidden=(256, 256), dropout=0.15, verbose=False):
    scores = []
    for fold in FOLDS:
        fold_train, fold_valid = make_fold_frames(fold)
        model, mu, sd, best_epoch = train_mlp_fold(
            fold_train, feature_cols, seed=seed, T_soft=T_soft, hidden=hidden, dropout=dropout, verbose=verbose,
        )
        pred = {g: predict_group_mlp(model, fold_valid, g, feature_cols, mu, sd) for g in TARGET_COLS}
        answer_df = make_answer_df(fold_valid)
        pred_df = make_pred_df(fold_valid, pred)
        score, _, _ = metric(answer_df, pred_df)
        scores.append(score)
        if verbose:
            print(f"  {fold['name']}: Score={score:.4f} (best_epoch={best_epoch})")
    return float(np.mean(scores)), scores

### 14-4. MLP 단독 baseline CV

기본값(`T_soft=0.006`, 은닉층 256-256, dropout 0.15)으로 먼저 돌려보고, CatBoost 쪽 기준점(9-2/9-3의 `baseline_mean`·`tau_seed_mean`)과 나란히 비교한다.

In [67]:
mlp_baseline_mean, mlp_baseline_scores = cv_score_mlp(verbose=True)
print(f"\nMLP baseline(T_soft=0.006) 3-fold 평균 Score: {mlp_baseline_mean:.4f}")
print(f"CatBoost 기본값(9절 이전) baseline_mean: {baseline_mean:.4f}")
print(f"CatBoost 최종안(τ=0.70+actual가중) tau_seed_mean: {tau_seed_mean:.4f}")

  epoch 0: val_score=0.4811 (best 0.4811 @ 0)
  epoch 20: val_score=0.5888 (best 0.5888 @ 20)
  epoch 40: val_score=0.5995 (best 0.6161 @ 36)
  epoch 60: val_score=0.6001 (best 0.6161 @ 36)
  fold1: Score=0.5968 (best_epoch=36)
  epoch 0: val_score=0.4623 (best 0.4623 @ 0)
  epoch 20: val_score=0.5976 (best 0.6007 @ 17)
  epoch 40: val_score=0.6040 (best 0.6071 @ 35)
  epoch 60: val_score=0.6055 (best 0.6098 @ 54)
  epoch 80: val_score=0.6082 (best 0.6124 @ 75)
  epoch 100: val_score=0.6121 (best 0.6126 @ 92)
  epoch 120: val_score=0.6130 (best 0.6144 @ 104)
  epoch 140: val_score=0.6084 (best 0.6155 @ 124)
  fold2: Score=0.6281 (best_epoch=124)
  epoch 0: val_score=0.4860 (best 0.4860 @ 0)
  epoch 20: val_score=0.6129 (best 0.6130 @ 19)
  epoch 40: val_score=0.6263 (best 0.6263 @ 40)
  epoch 60: val_score=0.6313 (best 0.6313 @ 60)
  epoch 80: val_score=0.6292 (best 0.6313 @ 60)
  fold3: Score=0.6507 (best_epoch=60)

MLP baseline(T_soft=0.006) 3-fold 평균 Score: 0.6252
CatBoost 기본값(9절 이전

### 14-5. T_soft 스윕

계단을 얼마나 부드럽게 근사할지 조절하는 온도(temperature) 하이퍼파라미터. 너무 작으면(진짜 계단에 가까움) 기울기가 날카로워져 학습이 불안정하고, 너무 크면 계단 구조 자체가 뭉개진다.

In [23]:
T_SOFT_GRID = [0.003, 0.006, 0.01, 0.02, 0.04]

t_soft_rows = []
for t_soft in T_SOFT_GRID:
    mean_score, fold_scores = cv_score_mlp(T_soft=t_soft)
    t_soft_rows.append({"T_soft": t_soft, "cv_mean_score": mean_score, "fold_scores": fold_scores})
    print(f"T_soft={t_soft}: 3-fold 평균 Score={mean_score:.4f} (fold별 {[round(s, 4) for s in fold_scores]})")

t_soft_df = pd.DataFrame(t_soft_rows).sort_values("cv_mean_score", ascending=False).reset_index(drop=True)
best_T_soft = t_soft_df.loc[0, "T_soft"]
print(f"\n최적 T_soft: {best_T_soft}")
t_soft_df

T_soft=0.003: 3-fold 평균 Score=0.6265 (fold별 [np.float64(0.5962), np.float64(0.6294), np.float64(0.6538)])
T_soft=0.006: 3-fold 평균 Score=0.6252 (fold별 [np.float64(0.5968), np.float64(0.6281), np.float64(0.6507)])
T_soft=0.01: 3-fold 평균 Score=0.6263 (fold별 [np.float64(0.5975), np.float64(0.63), np.float64(0.6512)])
T_soft=0.02: 3-fold 평균 Score=0.6264 (fold별 [np.float64(0.5979), np.float64(0.6288), np.float64(0.6526)])
T_soft=0.04: 3-fold 평균 Score=0.6250 (fold별 [np.float64(0.5964), np.float64(0.6256), np.float64(0.653)])

최적 T_soft: 0.003


,T_soft,cv_mean_score,fold_scores
0,0.003,0.626475,"[0.5962337750554235, 0.6293996603476725, 0.653..."
1,0.020,0.626447,"[0.5979486068326488, 0.6287650866201487, 0.652..."
2,0.010,0.626253,"[0.5975459877886944, 0.63004620870004, 0.65116..."
3,0.006,0.625204,"[0.5968416810853067, 0.6281042634751776, 0.650..."
4,0.040,0.624992,"[0.5963886385571744, 0.6256135752081093, 0.652..."


### 14-6. seed 안정성 확인

`model-tuning` 스킬 3절, 그리고 11~13절에서 계속 강조한 원칙 — 단일 seed 결과를 믿지 않는다. seed 3개(42/7/2024)로 최적 T_soft를 재검증한다.

In [69]:
mlp_seed_scores = []
for seed in [42, 7, 2024]:
    mean_score, fold_scores = cv_score_mlp(T_soft=best_T_soft, seed=seed)
    mlp_seed_scores.append(mean_score)
    print(f"seed={seed}: 3-fold 평균 Score={mean_score:.4f} (fold별 {[round(s, 4) for s in fold_scores]})")

mlp_seed_mean = float(np.mean(mlp_seed_scores))
mlp_seed_std = float(np.std(mlp_seed_scores))
print(f"\nMLP seed 3개 평균: {mlp_seed_mean:.4f}, 표준편차: {mlp_seed_std:.4f}")
print(f"CatBoost 최종안(tau_seed_mean={tau_seed_mean:.4f}) 대비: {mlp_seed_mean - tau_seed_mean:+.4f}")

seed=42: 3-fold 평균 Score=0.6265 (fold별 [np.float64(0.5962), np.float64(0.6294), np.float64(0.6538)])
seed=7: 3-fold 평균 Score=0.6254 (fold별 [np.float64(0.5996), np.float64(0.6283), np.float64(0.6484)])
seed=2024: 3-fold 평균 Score=0.6250 (fold별 [np.float64(0.5952), np.float64(0.6319), np.float64(0.6478)])

MLP seed 3개 평균: 0.6256, 표준편차: 0.0006
CatBoost 최종안(tau_seed_mean=0.6213) 대비: +0.0043


### 14-7. CatBoost + MLP 블렌드 — 전역 가중치

MLP 단독으로 CatBoost를 이기지 못하더라도, **서로 다른 종류의 오차**를 만들 가능성이 높다(트리의 축 정렬 분할 vs 신경망의 매끄러운 함수, `phase6_nn_metric_loss.md` §3-7 참고). 두 모델을 한 번씩만 학습해 예측을 캐시해두고, 블렌드 가중치는 그 예측을 재사용해 빠르게 스캔한다.

In [70]:
fold_preds_blend = {}
for fold in FOLDS:
    fold_train, fold_valid = make_fold_frames(fold)
    cat_model = train_fold_model(fold_train, DEFAULT_PARAMS, quantile_alpha=best_tau, use_sample_weight=True)
    mlp_model, mlp_mu, mlp_sd, _ = train_mlp_fold(fold_train, FEATURE_COLS, T_soft=best_T_soft)

    cat_pred = {g: predict_group(cat_model, fold_valid, g) for g in TARGET_COLS}
    mlp_pred = {g: predict_group_mlp(mlp_model, fold_valid, g, FEATURE_COLS, mlp_mu, mlp_sd) for g in TARGET_COLS}
    fold_preds_blend[fold["name"]] = {"fold_valid": fold_valid, "cat": cat_pred, "mlp": mlp_pred}

print("fold별 CatBoost·MLP 예측 캐시 완료")

BLEND_WEIGHTS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]  # 0.0=CatBoost만, 1.0=MLP만

blend_rows = []
for w in BLEND_WEIGHTS:
    fold_scores = []
    for fold in FOLDS:
        ctx = fold_preds_blend[fold["name"]]
        pred = {g: (1 - w) * ctx["cat"][g] + w * ctx["mlp"][g] for g in TARGET_COLS}
        answer_df = make_answer_df(ctx["fold_valid"])
        pred_df = make_pred_df(ctx["fold_valid"], pred)
        score, _, _ = metric(answer_df, pred_df)
        fold_scores.append(score)
    mean_score = float(np.mean(fold_scores))
    blend_rows.append({"mlp_weight": w, "cv_mean_score": mean_score, "fold_scores": fold_scores})
    print(f"w_mlp={w}: 3-fold 평균 Score={mean_score:.4f}")

blend_df = pd.DataFrame(blend_rows).sort_values("cv_mean_score", ascending=False).reset_index(drop=True)
best_blend_w = blend_df.loc[0, "mlp_weight"]
print(f"\n최고 전역 블렌드 가중치: w_mlp={best_blend_w} (CV {blend_df.loc[0, 'cv_mean_score']:.4f})")
blend_df

fold별 CatBoost·MLP 예측 캐시 완료
w_mlp=0.0: 3-fold 평균 Score=0.6219
w_mlp=0.2: 3-fold 평균 Score=0.6267
w_mlp=0.4: 3-fold 평균 Score=0.6288
w_mlp=0.5: 3-fold 평균 Score=0.6293
w_mlp=0.6: 3-fold 평균 Score=0.6291
w_mlp=0.8: 3-fold 평균 Score=0.6276
w_mlp=1.0: 3-fold 평균 Score=0.6265

최고 전역 블렌드 가중치: w_mlp=0.5 (CV 0.6293)


,mlp_weight,cv_mean_score,fold_scores
0,0.5,0.629251,"[0.6079286483930997, 0.6274382167505219, 0.652..."
1,0.6,0.629056,"[0.6051934853945806, 0.6285431268037309, 0.653..."
2,0.4,0.628754,"[0.6084662308091245, 0.6275877647724444, 0.650..."
3,0.8,0.627609,"[0.6010239097900129, 0.6275487507852775, 0.654..."
4,0.2,0.626674,"[0.6069298792805892, 0.6260186823133546, 0.647..."
5,1.0,0.626475,"[0.5962337750554235, 0.6293996603476725, 0.653..."
6,0.0,0.621867,"[0.6033057742646094, 0.6215657932243673, 0.640..."


### 14-8. 그룹별 블렌드 가중치

`total_score`는 그룹별 점수의 단순평균이라, 그룹마다 다른 블렌드 가중치를 써도 전체 점수를 정확히 최적화할 수 있다(11절에서 검산한 것과 같은 원리). 단, 여기서는 그룹 하나만 채점해야 하므로 `make_pred_df`(3개 그룹 컬럼을 다 요구함)를 쓰지 않고 `metric.py` 로직을 그 그룹에 대해서만 직접 복제한다. 14-7의 예측 캐시를 그대로 재사용한다.

In [71]:
GROUP_BLEND_GRID = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]


def single_group_score(fold_valid, g, pred_kwh):
    """단일 그룹만 metric.py 로직대로 직접 채점(다른 그룹 컬럼이 없어도 되도록 make_pred_df를 안 씀)."""
    actual = fold_valid[g].to_numpy(dtype=float)
    pred = np.asarray(pred_kwh, dtype=float)
    capacity = CAPACITY_KWH[g]

    valid = actual >= capacity * 0.10
    actual_v, pred_v = actual[valid], pred[valid]

    error_rate = np.abs(pred_v - actual_v) / capacity
    nmae = float(np.mean(error_rate))

    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    ficr = float(np.sum(actual_v * unit_price) / np.sum(actual_v * 4.0))
    return nmae, ficr


group_blend_best = {}
for g in TARGET_COLS:
    rows = []
    for w in GROUP_BLEND_GRID:
        nmae_list, ficr_list = [], []
        for fold in FOLDS:
            ctx = fold_preds_blend[fold["name"]]
            fold_valid = ctx["fold_valid"]
            pred_g = (1 - w) * ctx["cat"][g] + w * ctx["mlp"][g]
            nmae, ficr = single_group_score(fold_valid, g, pred_g)
            nmae_list.append(nmae)
            ficr_list.append(ficr)
        mean_nmae = float(np.mean(nmae_list))
        mean_ficr = float(np.mean(ficr_list))
        group_score = 0.5 * (1 - mean_nmae) + 0.5 * mean_ficr
        rows.append({"weight": w, "group_score": group_score, "nmae": mean_nmae, "ficr": mean_ficr})

    g_df = pd.DataFrame(rows).sort_values("group_score", ascending=False).reset_index(drop=True)
    group_blend_best[g] = g_df.loc[0, "weight"]
    print(f"{g}: 최적 w_mlp={g_df.loc[0, 'weight']} (그룹 점수 {g_df.loc[0, 'group_score']:.4f})")
    display(g_df)

print("\n그룹별 최적 가중치:", group_blend_best)

group_blend_fold_scores = []
for fold in FOLDS:
    ctx = fold_preds_blend[fold["name"]]
    pred = {g: (1 - group_blend_best[g]) * ctx["cat"][g] + group_blend_best[g] * ctx["mlp"][g] for g in TARGET_COLS}
    answer_df = make_answer_df(ctx["fold_valid"])
    pred_df = make_pred_df(ctx["fold_valid"], pred)
    score, _, _ = metric(answer_df, pred_df)
    group_blend_fold_scores.append(score)

group_blend_mean = float(np.mean(group_blend_fold_scores))
print(f"\n그룹별 최적 가중치 조합 3-fold 평균 Score: {group_blend_mean:.4f}")
print(f"전역 최적 가중치({best_blend_w}) 대비: {group_blend_mean - blend_df.loc[0, 'cv_mean_score']:+.4f}")
print(f"CatBoost 단독(w=0) 대비: {group_blend_mean - blend_df.loc[blend_df['mlp_weight'] == 0.0, 'cv_mean_score'].iloc[0]:+.4f}")

kpx_group_1: 최적 w_mlp=0.4 (그룹 점수 0.6475)


,weight,group_score,nmae,ficr
0,0.4,0.647474,0.126142,0.421089
1,0.2,0.646723,0.128624,0.422070
2,0.5,0.646130,0.125305,0.417564
3,0.6,0.645749,0.124764,0.416261
4,0.8,0.643333,0.124607,0.411273
5,0.0,0.642111,0.132104,0.416326
6,1.0,0.641754,0.125502,0.409010


kpx_group_2: 최적 w_mlp=0.5 (그룹 점수 0.6520)


,weight,group_score,nmae,ficr
0,0.5,0.651998,0.133702,0.437697
1,0.6,0.651190,0.134085,0.436466
2,0.4,0.650852,0.133618,0.435323
3,0.8,0.648883,0.135720,0.433486
4,0.2,0.648339,0.134390,0.431068
5,1.0,0.645295,0.138344,0.428934
6,0.0,0.642754,0.136232,0.421739


kpx_group_3: 최적 w_mlp=1.0 (그룹 점수 0.5924)


,weight,group_score,nmae,ficr
0,1.0,0.592377,0.150928,0.335682
1,0.8,0.590612,0.149547,0.330770
2,0.6,0.590228,0.149037,0.329492
3,0.5,0.589625,0.149129,0.328379
4,0.4,0.587937,0.149406,0.325279
5,0.2,0.584959,0.150462,0.320380
6,0.0,0.580639,0.152125,0.313404



그룹별 최적 가중치: {'kpx_group_1': np.float64(0.4), 'kpx_group_2': np.float64(0.5), 'kpx_group_3': np.float64(1.0)}

그룹별 최적 가중치 조합 3-fold 평균 Score: 0.6306
전역 최적 가중치(0.5) 대비: +0.0014
CatBoost 단독(w=0) 대비: +0.0087


### 14-9. 최종 블렌드 seed 3개 재검증

11~13절 교훈 — 단일 seed(단일 CatBoost seed + 단일 MLP seed 조합)에서 나온 개선은 믿지 않는다. 그룹별 최적 가중치(14-8)와 전역 최적 가중치(14-7) 중 더 나은 쪽을, CatBoost·MLP seed를 함께 바꿔가며 재검증한다.

In [72]:
use_group_blend = group_blend_mean >= blend_df.loc[0, "cv_mean_score"]
final_weights = group_blend_best if use_group_blend else {g: best_blend_w for g in TARGET_COLS}
print(f"seed 재검증 대상: {'그룹별 가중치' if use_group_blend else '전역 가중치'} {final_weights}")

blend_seed_scores = []
for seed in [42, 7, 2024]:
    fold_scores = []
    for fold in FOLDS:
        fold_train, fold_valid = make_fold_frames(fold)
        cat_model = train_fold_model(fold_train, DEFAULT_PARAMS, quantile_alpha=best_tau, use_sample_weight=True, seed=seed)
        mlp_model, mlp_mu, mlp_sd, _ = train_mlp_fold(fold_train, FEATURE_COLS, T_soft=best_T_soft, seed=seed)

        pred = {}
        for g in TARGET_COLS:
            cat_pred_g = predict_group(cat_model, fold_valid, g)
            mlp_pred_g = predict_group_mlp(mlp_model, fold_valid, g, FEATURE_COLS, mlp_mu, mlp_sd)
            w = final_weights[g]
            pred[g] = (1 - w) * cat_pred_g + w * mlp_pred_g

        answer_df = make_answer_df(fold_valid)
        pred_df = make_pred_df(fold_valid, pred)
        score, _, _ = metric(answer_df, pred_df)
        fold_scores.append(score)

    seed_mean = float(np.mean(fold_scores))
    blend_seed_scores.append(seed_mean)
    print(f"seed={seed}: 3-fold 평균 Score={seed_mean:.4f} (fold별 {[round(s, 4) for s in fold_scores]})")

blend_seed_mean = float(np.mean(blend_seed_scores))
blend_seed_std = float(np.std(blend_seed_scores))
print(f"\n블렌드 seed 3개 평균: {blend_seed_mean:.4f}, 표준편차: {blend_seed_std:.4f}")
print(f"CatBoost 단독 최종안(tau_seed_mean={tau_seed_mean:.4f}) 대비 개선폭: {blend_seed_mean - tau_seed_mean:+.4f} (표준편차의 {(blend_seed_mean - tau_seed_mean) / blend_seed_std if blend_seed_std > 0 else float('nan'):.1f}배)")

seed 재검증 대상: 그룹별 가중치 {'kpx_group_1': np.float64(0.4), 'kpx_group_2': np.float64(0.5), 'kpx_group_3': np.float64(1.0)}
seed=42: 3-fold 평균 Score=0.6306 (fold별 [np.float64(0.6061), np.float64(0.6317), np.float64(0.654)])
seed=7: 3-fold 평균 Score=0.6302 (fold별 [np.float64(0.607), np.float64(0.6338), np.float64(0.6497)])
seed=2024: 3-fold 평균 Score=0.6276 (fold별 [np.float64(0.6028), np.float64(0.635), np.float64(0.6451)])

블렌드 seed 3개 평균: 0.6295, 표준편차: 0.0013
CatBoost 단독 최종안(tau_seed_mean=0.6213) 대비 개선폭: +0.0082 (표준편차의 6.2배)


### 14-10. 종합 해석

**14-4, MLP 단독 baseline**: T_soft=0.006 기준 3-fold 평균 0.6252 — **CatBoost 최종안(τ=0.70+actual가중, 0.6213)을 이미 넘어섬**, CatBoost 기본값(0.5908) 대비로는 압도적. fold별로 학습 곡선을 보면(에폭 로그) 100~150 에폭 근방에서 수렴하는 경우가 많아 `max_epochs=300`이 여유 있게 충분했다.

**14-5, T_soft 스윕**: 0.003(0.6265)~0.02(0.6264) 구간은 사실상 평평하고, 0.006(baseline, 0.6252)도 그 범위 안에 있다. 0.04에서만 소폭 하락(0.6250). 즉 이 하이퍼파라미터는 민감하지 않다 — 극단으로 가지만 않으면 성능이 크게 흔들리지 않는다는 뜻이라 안정적인 신호로 판단한다.

**14-6, seed 안정성(핵심)**: T_soft=0.003으로 seed 3개(42/7/2024) 재검증 → 평균 **0.6256, 표준편차 0.0006**(매우 작음). 11~13절에서 계속 "단일 seed 개선이 재검증하면 사라졌던" 것과 정반대로, **여기서는 개선폭(+0.0043)이 표준편차보다 훨씬 크게(7배) 유지된다.**

**14-7, CatBoost+MLP 전역 블렌드**: 가중치 0.5(둘을 반반 섞음)에서 3-fold 평균 **0.6293** — MLP 단독(0.6265, w=1.0)보다도 더 좋다. 이건 단순히 "MLP가 CatBoost보다 낫다"가 아니라 **두 모델이 서로 다른 종류의 오차를 만들어 섞으면 서로 상쇄된다**는 뜻이다(트리의 축 정렬 분할 vs 신경망의 매끄러운 함수 근사 — 민석님이 공유한 외부 문서 phase6 §3-7과 같은 메커니즘).

**14-8, 그룹별 블렌드**: group_1=0.4, group_2=0.5, **group_3=1.0(MLP만, CatBoost 완전 배제)**. group_3은 11절부터 계속 우리 모델의 약점이었던 그룹인데, 여기서 "CatBoost를 아예 안 쓰는 게 최적"이라는 결과가 나온 것이 의미심장하다 — group_3은 라벨 기간이 짧아(2023년부터) 데이터가 적은데, MLP의 매끄러운 함수 근사가 이런 소규모·저신호 상황에서 트리의 축 정렬 분할보다 덜 과적합했을 가능성이 있다. 그룹별 조합 3-fold 평균 **0.6306**(전역 블렌드 대비 +0.0014).

**14-9, 최종 검증(가장 중요)**: 그룹별 최적 가중치를 CatBoost·MLP seed를 함께 바꿔가며(42/7/2024) 재검증 → 평균 **0.6295, 표준편차 0.0013**. **CatBoost 단독 최종안(tau_seed_mean=0.6213) 대비 개선폭 +0.0082, 표준편차의 6.2배.** 11~13절의 실패 패턴(개선폭이 표준편차 이내로 사라짐)과 뚜렷하게 다르다 — 이번 개선은 진짜다.

**결론**: 민석님이 공유해주신 외부 파이프라인의 핵심 통찰(미분 가능한 FICR 손실로 채점 산식을 직접 최적화)이 우리 파이프라인에도 그대로 재현됐다. **최종 채택안: CatBoost(τ=0.70+actual가중) + MLP(T_soft=0.003) 블렌드, 그룹별 가중치(group_1=0.4/group_2=0.5/group_3=1.0)**. 다음 단계는 이 구조를 `train.ipynb`/`inference.ipynb`에 반영해 전체 데이터로 재학습하고 재제출하는 것이다.

## 15. 그룹별 τ 재검토 (CatBoost)

**배경**: 2차 제출(리더보드 Score 0.625596) 이후 외부 참고 파이프라인(exp017, Public 0.63886)과의 격차를 다시 뜯어보다가, 그쪽은 `phase4_tuning.md` §2-4에서 그룹마다 **다른** τ(group_1=0.70, group_2=0.50, group_3=0.65)를 썼다는 걸 확인했다. group_2는 세 그룹 중 이용률이 가장 높아(0.328) 발전량이 10% 임계값 밑으로 떨어질 확률이 낮으므로, 예측을 위쪽으로 편향시키는 τ>0.5의 필요성이 상대적으로 적었다는 게 그 근거다.

**우리가 아직 안 해본 것**: 9절에서 τ=0.70(전 그룹 공통)을 확정했고, 11-4에서 group_3만 다른 τ를 시도한 적은 있지만(seed 재검증에서 개선폭 0으로 결론) — 11절은 애초에 "group_3 전용 대응"이라는 좁은 목적으로 설계됐다. **group_1·group_2는 지금까지 한 번도 다른 τ를 시도한 적이 없다.** 이번엔 3개 그룹을 모두 독립적으로 검토한다.

**절차**: (1) 그룹마다 독립적으로 τ 그리드(0.50~0.75)를 탐색해 그룹별 최적 τ를 찾는다(11-4와 같은 방식 — 04번에서 확인한 풀링 구조의 이점은 유지하고, 예측만 그 그룹에 한해 다른 τ 모델 것으로 교체). (2) 그룹별 최적 τ를 하나의 조합으로 합쳐 CV·seed로 재검증한다. (3) 개선이 확인되면, 14절의 CatBoost+MLP 그룹별 블렌드를 이 새로운 CatBoost(그룹별 τ)로 다시 계산해 블렌드 가중치를 재탐색한다.

**leakage 체크**(`leakage-guard` 스킬): τ는 학습 시 손실 함수의 파라미터일 뿐, 예측기준시점에 없는 정보를 끌어오지 않는다 — 9절에서 이미 확인한 것과 동일하게 안전하다.


*(이 절은 실패/미채택으로 결론나 코드는 삭제하고 마크다운 기록만 남겼다 — 실행 결과는 각 소절 설명과 마지막 종합 해석, `reports/05_tuning.md`를 참고.)*

### 15-1. 그룹별 τ=0.70(공통) baseline 캐시

11-3~11-4와 같은 방식 — fold마다 baseline 모델(τ=`best_tau`=0.70, actual 가중)을 한 번씩 학습해 그룹별 예측을 캐시해둔다. 15-2에서 그룹마다 다른 τ 후보를 시도할 때, 그 그룹 예측만 교체하고 나머지 그룹은 이 baseline 예측을 그대로 쓴다.


### 15-2. 그룹별 독립 τ 그리드 탐색

그룹 g마다: 후보 τ로 새로 학습한 모델의 예측을 그 그룹에만 적용하고, 나머지 두 그룹은 baseline(τ=0.70) 예측을 그대로 둔 채 전체 Score를 재계산한다(11-4와 동일한 방식). 이렇게 하면 "통합모델이 그룹별모델보다 낫다"는 04번 풀링 효과를 잃지 않으면서 그룹별 τ 효과만 순수하게 볼 수 있다.

**실행 시간 참고**: τ 후보 5개(0.70은 baseline 재사용) × 그룹 3개 × 3-fold = 45번 재학습이라 9-2/11-4보다 오래 걸린다. 시간이 부족하면 `TAU_GROUP_GRID`를 줄여도 된다.


### 15-3. 그룹별 최적 τ 조합을 하나로 합쳐 CV 재확인

15-2는 그룹 하나만 바꾸고 나머지 둘은 baseline에 고정한 "개별 효과"였다. 이번엔 `best_tau_per_group`에 담긴 그룹별 τ를 **동시에** 적용했을 때도 개선이 유지되는지 확인한다(그룹 간 상호작용으로 개별 효과의 단순 합과 다를 수 있다). 같은 τ를 쓰는 그룹끼리는 모델을 공유해 재학습 횟수를 줄인다.


### 15-4. seed 안정성 재검증

`model-tuning` 스킬 3절, 그리고 11~13절에서 거듭 확인한 원칙 — 단일 seed 개선은 믿지 않는다. `best_tau_per_group` 조합을 seed 3개(42/7/2024)로 재검증한다.


### 15-5. CatBoost+MLP 블렌드 가중치 재탐색 (조건부)

15-4에서 그룹별 τ 조합이 baseline(공통 τ=0.70)을 표준편차 이상으로 못 넘으면, 14절의 블렌드 가중치(`group_blend_best`)를 다시 계산할 이유가 없다 — 그래서 개선이 확인됐을 때만 자동으로 진행한다(민석님이 매번 판단할 필요 없이 코드가 스스로 건너뛴다, 12-3/13-3과 같은 원칙). 개선이 확인되면 14-7·14-8과 같은 절차를 CatBoost 쪽만 `best_tau_per_group`로 바꿔서 반복한다.


### 15-6. 블렌드 seed 재검증 (조건부)

15-5에서 재탐색이 실제로 실행됐을 때만(`RETUNE_BLEND`가 True일 때만) 14-9와 같은 방식으로 CatBoost·MLP seed를 함께 바꿔가며 최종 확인한다.


### 15-7. 종합 해석

**15-2, 그룹별 독립 τ 그리드**: group_1·group_2는 baseline(τ=0.70)이 그대로 최적이었다(둘 다 개선폭 +0.0000) — 그룹별로 다르게 줄 여지 자체가 없었다는 뜻. group_3만 τ=0.75에서 +0.0005 개선됐는데, 이는 11-4에서 이미 확인했던 것과 정확히 같은 크기·같은 방향의 신호다.

**15-3·15-4, 조합 CV와 seed 재검증(핵심)**: 세 그룹을 동시에 바꾼 조합(g1=0.70/g2=0.70/g3=0.75)의 단일 seed CV는 0.6224(+0.0011)로 보였지만, seed 3개(42/7/2024)로 재검증하니 평균 0.6209로 오히려 baseline(tau_seed_mean=0.6213)보다 **-0.0004 낮았다** — 표준편차(0.0013)의 -0.3배, 즉 개선은커녕 방향조차 불확실한 순수 노이즈다. 11-4(group_3 단독)에서 내렸던 결론("+0.0005~0.0006은 seed 노이즈")이 이번엔 3개 그룹을 함께 봐도 그대로 재확인됐다.

**15-5·15-6**: 조합이 baseline을 표준편차 이상으로 못 넘었으므로 코드가 자동으로 블렌드 재탐색을 건너뛰었다 — 14절 채택안(`group_blend_best` = g1:0.4/g2:0.5/g3:1.0, `blend_seed_mean`=0.6295)이 그대로 최종안이다.

**외부 파이프라인과의 격차 재해석**: 이번 실험은 "베이스 GBDT가 그룹별 τ로 학습돼 있느냐(외부는 그렇고, 우리는 공통 τ=0.70이었다)"가 블렌드 개선폭 차이(우리 +0.0030 vs 외부 +0.0160, 2026-07-23 세션 격차 재분석)의 원인일 수 있다는 가설을 검증한 것이었다. **결과는 이 가설을 기각한다** — 우리 데이터·CV 구조에서는 그룹별 τ 자체가 유의미한 개선을 내지 못하므로, 베이스 τ 구조 차이가 격차의 핵심 원인일 가능성은 낮다. 남은 유력 후보는 그때 3순위였던 **피처셋 격차(179개 vs 50개, 특히 SCADA 기반 물리 파워커브 예측치 `pc_pred_*`)** 쪽으로 좁혀진다.

**결론**: τ=0.70(전 그룹 공통) + actual 가중을 최종안으로 유지한다. CatBoost+MLP 그룹별 블렌드(g1=0.4/g2=0.5/g3=1.0)도 변경 없음. 그룹별 τ 실험은 여기서 마무리하고, 다음 우선순위는 피처셋 격차 검토로 이동한다.


## 16. SCADA 기반 물리 파워커브 피처 (간소화 버전)

**배경**: 15절(그룹별 τ 재검토)이 개선 없음으로 끝나면서, 외부 파이프라인(exp017, Public 0.63886)과의 격차 원인 후보 중 "베이스 τ 구조 차이"는 기각됐다(`HANDOFF.md`·메모리 참고). 남은 유력 후보는 **피처셋 격차**다 — 외부는 SCADA로 학습한 물리 파워커브 예측치(`pc_pred_*`)를 피처로 써서 feature importance의 23.6%를 차지할 만큼 큰 효과를 봤는데(외부 문서 phase3 §3-7), 우리 50개 피처엔 이런 물리+ML 하이브리드 피처가 없다.

**leakage-guard 판정**: `03_features.ipynb` 작업 당시 이미 "SCADA 파워커브는 target-encoding 유사 리스크로 이번 버전에 포함하지 않음 — 필요 시 CV fold 내부 적합 전제로 별도 검토"라고 명시해뒀다(`reports/03_features.md`). target encoding(카테고리별 평균을 그대로 lookup)과 달리 파워커브는 "풍속(연속값) → kWh"라는 매끄러운 물리 함수를 수십 개 구간으로 근사하는 것이라 자유도가 훨씬 낮지만, 그래도 타깃 정보가 피처에 녹아드는 건 맞다. 그래서 **3-fold 각각의 학습 구간 SCADA로만 곡선을 다시 적합**하고(전체 라벨 기간으로 한 번에 적합하면 검증 구간 정보가 새어 들어간다), 검증·테스트 구간엔 이미 안전한 예보 풍속 피처(`{group}_ldaps_ws10m`)를 그 곡선에 대입만 한다 — 이는 `leakage-guard` 스킬이 명시적으로 허용하는 "전처리 파이프라인(스케일러 등)은 분리 후 train 파트에만 fit" 원칙과 같은 구조다.

**이번 버전(간소화, 민석님 선택)**: 외부처럼 "예보 풍속 → SCADA 나셀풍속" 편향보정은 넣지 않는다 — 외부도 이 보정(Î± 역산)을 자체적으로 시도했다가 물리적으로 불가능한 값이 나와 버렸다고 기록돼 있다(`phase2_features.md` §2-7). 이미 있는 그룹가중 LDAPS 10m 풍속(우리 50개 피처 중 하나, `phase1 EDA` 근거상 가장 상관이 높은 원시 풍속)을 그대로 파워커브의 입력으로 쓴다. 효과가 있으면 그때 허브고도 외삽·편향보정 등 정교화를 검토한다.


*(이 절은 실패/미채택으로 결론나 코드는 삭제하고 마크다운 기록만 남겼다 — 실행 결과는 각 소절 설명과 마지막 종합 해석, `reports/05_tuning.md`를 참고.)*

### 16-1. SCADA 시간 집계 + 라벨 상관 재현 검산

`02_eda.ipynb` 4-0·4-4절과 완전히 같은 방식(VESTAS 극단 센서오류만 설비용량×1.5 초과 시 NaN 처리, `power_kw10m`을 10분 에너지로 보고 시간당 6개 합산)으로 그룹별 시간당 SCADA 발전량을 다시 만든다. 라벨과의 상관이 그때 기록한 값(0.9998/0.9998/0.9966)과 같게 나오는지 먼저 확인해, 아래 파워커브 적합에 쓸 SCADA가 신뢰할 만한지 검증한다.


### 16-2. fold-safe 파워커브 적합 함수

그룹·fold마다: 그 fold의 **학습 구간 SCADA**만으로 (풍속, 발전량) 쌍을 0.5 m/s 구간으로 나눠 구간 중앙값을 구하고, 등장회귀(isotonic regression — "풍속이 늘면 발전량도 늘거나 같아야 한다"는 물리적 단조 제약을 강제하는 회귀)로 매끄럽게 잇는다. 표본이 20개 미만인 구간은 노이즈로 보고 제외한다. `sklearn.isotonic.IsotonicRegression(y_min=0, y_max=설비용량, out_of_bounds="clip")`을 쓰면 구간 밖 입력(아주 낮거나 높은 풍속)도 0~설비용량 범위 안에서 자연스럽게 처리된다.


### 16-3. 곡선 모양 확인

fold3(가장 데이터가 많음)의 학습 구간으로 적합한 곡선이 물리적으로 말이 되는지 몇 개 풍속 지점에서 확인한다 — 저풍속에서 0에 가깝고, 올라가다가 정격 근처에서 설비용량에 가깝게 평평해져야 한다(파워커브 이론).


### 16-4. CV 비교 (baseline vs +pc_pred)

fold마다 16-2의 함수로 그룹별 곡선을 새로 적합해 `{group}_pc_pred` 3개 컬럼을 추가한 피처셋(53개)과 baseline(50개)을 같은 3-fold CV(τ=`best_tau`, actual 가중)로 비교한다. `cv_score_ext`(9절)를 baseline 계산에 그대로 재사용한다.


### 16-5. seed 재검증 (조건부)

12·13·15절과 같은 원칙 — baseline을 못 이기면 여기서 "개선 없음"으로 정직하게 마무리한다.


### 16-6. 종합 해석

**16-1, 라벨 상관 재현**: group_1 0.9998, group_2 0.9998, group_3 0.9966 — `02_eda.ipynb`에서 기록한 값과 소수점 4자리까지 정확히 일치. SCADA 시간 집계 로직을 신뢰할 수 있다는 뜻이고, 이후 파워커브 적합의 입력 데이터는 문제가 없다.

**16-3, 곡선 모양**: 세 그룹 모두 cut-in(2~4 m/s 부근에서 값이 붙기 시작) → 정격 근처(10~12 m/s)에서 평평해지는 전형적인 S자 파워커브가 나왔다. 다만 최댓값이 설비용량에 못 미친다(group_1 16,570/21,600=76.7%, group_2 18,308/21,600=84.8%, group_3 16,462/21,000=78.4%) — 이는 이상하지 않다. 그룹 안 6대(5대) 터빈이 동시에 전부 정격 출력을 내는 순간은 드물기 때문에 그룹 합계 곡선의 상한이 이론적 설비용량보다 낮게 나오는 게 정상이다.

**16-4, CV 비교(핵심)**: baseline(50개, 0.6219) vs +pc_pred(53개, 0.6217) — **개선폭 -0.0002로 사실상 없음(오히려 미세하게 손해)**. seed 재검증까지 갈 필요 없이 16-5가 자동으로 "개선 없음"으로 종료했다.

**왜 실패했는가**: 이 버전의 `pc_pred`는 이미 피처셋에 있는 `{group}_ldaps_ws10m` **단 하나의 변수를 단조 변환(isotonic)한 것**에 불과하다. CatBoost는 이미 같은 변수의 원본·제곱·세제곱(`ws10m`/`ws10m_sq`/`ws10m_cube`)을 갖고 있어서, 트리 분할만으로도 "풍속이 오르면 발전량도 오르다가 어느 지점에서 꺾인다"는 단조·포화 관계를 스스로 근사할 수 있다 — 11·12·13절에서 반복됐던 "이미 모델이 아는 정보의 재포장" 패턴과 정확히 같다.

**외부 파이프라인과의 진짜 차이 재조명**: 외부 문서(`phase2_features.md` §3-1)를 다시 보면, 그쪽 `pc_pred`가 강력했던 진짜 이유는 파워커브 변환 자체가 아니라 그 **입력 풍속을 여러 NWP 소스(LDAPS 평균·LDAPS 최고상관격자·GFS 평균)의 회귀 조합으로 SCADA 나셀풍속에 맞춰 보정**했다는 데 있다 — 그 보정으로 RMSE가 3.9→1.8 m/s로 절반 넘게 줄었다고 명시돼 있다. 즉 "풍속 오차 자체를 줄이는 것"이 핵심이었고, 우리가 이번에 뺀 게 정확히 그 부분이다. 파워커브 변환은 그 위에 얹힌 부차적 단계였을 가능성이 높다.

**결론**: 이번 간소화 버전(단일 풍속 소스 + 파워커브 변환만)은 채택하지 않는다. 전체 설계(다중 NWP 소스 회귀 보정 + 파워커브)를 시도하려면 상당한 추가 코드(SCADA 나셀풍속 시간평균 재구성, 여러 풍속 피처 간 회귀, fold별 재적합)가 필요하다 — 투자 대비 효과는 아직 검증되지 않았다.


## 17. SCADA 다중 NWP 소스 회귀 편향보정 + 파워커브 (전체 설계)

**배경**: 16절(간소화 버전, LDAPS 10m 풍속을 그대로 파워커브 입력으로 사용)이 개선 없음(-0.0002)으로 끝났다. 16-5 해석에서 "외부의 `pc_pred`가 강력했던 진짜 이유는 파워커브 변환 자체가 아니라 그 앞단의 **다중 NWP 소스 회귀로 SCADA 나셀풍속을 보정하는 단계**(외부 기록 RMSE 3.9→1.8 m/s)였을 가능성"이 제기됐다 — 민석님이 "간소화 대신 전체 설계로 해보자"고 결정해서 이번 절에서 그 가설을 직접 구현·검증한다.

**무엇을 하는가(결론 먼저)**: 지금까지 파워커브 입력은 `{group}_ldaps_ws10m`(예보 풍속 하나) 그대로였다. 이번엔 그 대신 **GFS(10m/80m/100m/850hPa)와 LDAPS(10m)를 합쳐 회귀로 SCADA 나셀풍속(터빈이 실제로 맞은 바람)을 추정한 값**을 파워커브 입력으로 쓴다. 여기에 더해, 외부 파이프라인의 feature importance 표를 재분석해보니 1위 피처가 `pc_pred` 원본이 아니라 **5시간 이동평균**이었다는 걸 확인해(전체 중요도 15.7%, 2위의 4배), 이번 절에서는 파워커브 값의 시간 평활화까지 함께 검증한다.

**leakage-guard 판정**: 풍속 보정 회귀·파워커브·이동평균 모두 각 fold의 학습 구간 데이터로만 만들고(전처리는 train에만 fit 원칙), 검증·테스트 구간엔 이미 안전한 예보 피처만 대입한다. 이동평균은 과거 방향(trailing)만 보므로 미래 정보가 섞이지 않는다.

**이번 절의 최소 의존성**: 0~4절(셋업) + 9절 일부(`best_tau`/`DEFAULT_PARAMS`/`cv_score_ext`/`train_and_score_fold_ext`/`make_fold_frames`/`FOLDS`) — 16절을 실행하지 않아도 단독으로 실행 가능하도록 SCADA 로딩부터 다시 한다.

*(이 절은 실패/미채택으로 결론나 코드는 삭제하고 마크다운 기록만 남겼다 — 실행 결과는 각 소절 설명과 마지막 종합 해석, `reports/05_tuning.md`를 참고.)*

### 17-1. SCADA 시간 집계 — 발전량(kWh)과 나셀풍속(m/s) 동시 재구성

16-1과 같은 방식(VESTAS 극단 센서오류는 설비용량×1.5 초과 시 NaN 처리, `power_kw10m`을 10분 에너지로 보고 시간당 6개 합산)으로 그룹별 시간당 SCADA 발전량을 만든다. 이번엔 **풍속 컬럼(`_ws`)도 같은 시간 창으로 평균**해서 그룹별 시간당 나셀풍속을 추가로 만든다 — 이게 이번 절 회귀의 타깃(정답)이다. 라벨 상관(0.9998/0.9998/0.9966)이 재현되는지 먼저 검산해 SCADA 신뢰도를 확인하고, 나셀풍속 분포도 물리적으로 말이 되는지(음수 없음, 대략 0~20 m/s 범위) 확인한다.

### 17-2. fold-safe 다중 NWP 소스 회귀(편향보정) 함수

회귀 입력(피처)은 이미 있는 예보 풍속 5개 — GFS 10m/80m/100m/850hPa(그룹 공유 격자 1개로 수렴된 값)와 그룹별 LDAPS 10m. 타깃은 17-1에서 만든 그룹별 SCADA 시간당 나셀풍속이다. 표준화(StandardScaler — 평균 0, 표준편차 1로 맞추는 것, train으로만 fit)를 거친 뒤 릿지회귀(Ridge — 입력들이 서로 비슷해서(공선성) 생기는 회귀계수 불안정을 "너무 큰 계수에 벌점을 매겨서" 완화한 선형회귀)로 SCADA 나셀풍속을 예측한다. **각 fold의 학습 구간 데이터로만 학습**해서 fold-safe하게 만든다(16-2의 등장회귀와 같은 원칙).

### 17-3. 보정 효과 확인 — 학습 구간과 검증 구간(held-out) 모두

가장 데이터가 많은 fold3로 회귀를 적합해, 보정 전(원시 LDAPS 10m 풍속) 대비 보정 후 RMSE가 실제로 줄어드는지 본다. 이때 **학습 구간(fold_train)만 보면 안 된다** — "그 데이터로 학습했으니 잘 맞는 게 당연한" 착시(과적합)일 수 있다. 그래서 **회귀 학습에 전혀 쓰이지 않은 검증 구간(fold_valid)**에서도 RMSE를 따로 계산해 나란히 비교한다. 검증 구간에서도 원시 풍속보다 낮게 나오면 "진짜 신호를 잡은 것"이고, 학습 구간에서만 좋고 검증 구간에서 나빠지면 "과적합"이라는 뜻이다.

### 17-4. 보정 풍속 기반 파워커브 (fold-safe)

16-2와 동일한 등장회귀(isotonic regression — "풍속이 늘면 발전량도 늘거나 같아야 한다"는 물리적 단조 제약을 강제하는 회귀) 파워커브를, 입력만 원시 LDAPS 10m 대신 **보정 풍속**(`{group}_ws_corrected`)으로 바꿔 적합한다. 0.5m/s 구간, 표본 20개 미만 제외, `IsotonicRegression(y_min=0, y_max=설비용량)`으로 16-2와 구조가 같다.

### 17-5. 파워커브 값의 시간 평활화(5시간 이동평균) 피처 추가

외부 파이프라인의 feature importance 표를 보면 1위가 `pc_pred` 원본이 아니라 **5시간 이동평균**이었다(전체 중요도 15.7%, 2위의 4배). 이 점에 착안해 `{group}_pc_pred_v2`에 **과거 방향(trailing)** 이동평균을 추가한다. `pandas.rolling`의 기본 동작이 과거 방향이라 미래 시각을 참조하지 않으므로 leakage-guard 원칙(과거 방향 집계만 허용)에 어긋나지 않는다. fold_train/fold_valid 각각 시간순 정렬 후 독립적으로 계산한다(검증 구간 맨 앞 몇 시간은 창이 짧게 적용되지만 표본 수 대비 무시할 수준).

### 17-6. CV 비교 — baseline vs raw vs 이동평균 vs 둘 다

fold마다 (1) 학습 구간 SCADA로 풍속 보정 회귀 적합 → (2) 학습·검증 양쪽에 보정풍속 대입 → (3) 학습 구간 SCADA로 파워커브 적합 → (4) 학습·검증 양쪽에 `pc_pred_v2`와 그 이동평균 대입, 순서로 진행한다. baseline(50개) / +raw만(53개) / +이동평균만(53개) / +둘다(56개) 4개 조합을 같은 3-fold CV(τ=`best_tau`, actual 가중, 9절의 `cv_score_ext`/`train_and_score_fold_ext` 재사용)로 비교한다.

### 17-7. seed 재검증 (조건부)

11~16절과 같은 원칙: 4개 조합 중 baseline을 못 넘으면 seed를 바꿔볼 필요도 없이 여기서 끝난다(개선 없음으로 결론). 넘었을 때만(코드 자동 판단) seed 3개(42/7/2024)로 재검증해 노이즈인지 진짜 개선인지 가른다.

### 17-8. 종합 해석

**17-1, SCADA 검산**: 라벨 상관 0.9998/0.9998/0.9966 재현 -- 데이터는 신뢰할 수 있다.

**17-3, 풍속 보정(핵심 진단)**: 학습·검증 구간 모두에서 보정 풍속이 원시 풍속보다 뚜렷이 정확하다(검증 구간 RMSE: group_1 2.71->1.69, group_2 3.07->1.79, group_3 2.02->1.60 m/s -- 외부 기록 3.9->1.8과 방향·크기 일치). **회귀 자체는 완전히 성공했다.**

**17-6, CV 비교(최종)**:

| 조합 | CV | baseline 대비 |
|---|---:|---:|
| baseline(50개) | 0.6219 | 0 |
| +raw(53개) | 0.6204 | -0.0014 |
| +roll5 이동평균만(53개) | 0.6209 | -0.0010 |
| +raw+roll5(56개) | 0.6215 | **-0.0004** |

**네 조합 모두 baseline을 못 넘었다.** 17-7은 자동으로 seed 재검증을 건너뛰었다.

**그런데 방향성은 뚜렷하다**: raw만 넣었을 때(-0.0014)보다 이동평균을 넣었을 때(-0.0010), 그리고 raw+이동평균을 같이 넣었을 때(-0.0004)가 순서대로 baseline에 가까워진다 -- 손해가 71%(-0.0014 -> -0.0004)나 줄었다. 즉 **"외부의 5시간 이동평균이 핵심이었다"는 가설은 방향은 맞았다** -- 하지만 그 개선분으로는 raw 버전이 만든 손해를 완전히 메우지 못했다.

**결론**: 이 결과로 "다중 NWP 소스 회귀 편향보정 + 파워커브(원본이든 이동평균이든)"를 우리 파이프라인(50개 curated 피처 + τ=0.70 분위수회귀 + actual 가중)에 얹는 시도는 **여기서 마무리한다** -- 채택하지 않음, baseline 그대로 유지. 11(손실·가중치)·12(상호작용)·13(XBLWS/YBLWS)·16(파워커브 간소화)·17-raw·17-이동평균, 성격이 다른 6번의 시도가 전부 실패했다. 유일한 성공은 14절(MLP + 미분가능 FICR손실)뿐이다.

**왜 외부는 되고 우리는 안 됐는가(최종 해석)**: 회귀·이동평균 자체는 둘 다 옳았다는 게 이번에 증명됐다. 안 맞은 건 "이 신호를 어디에 얹느냐"다. 외부는 `pc_pred`류 피처를 179개 원시 피처의 출발점에(피처 선택 이전에) 넣고 그 위에서 전체를 다시 골랐다. 우리는 이미 여러 라운드의 feature-selection·quantile 회귀·actual 가중으로 세밀하게 맞춰진 50개 피처 세트 위에 얹었다 -- 이미 성숙한 모델일수록 새 피처 하나가 끼어들 여지(분할 구조를 흔들지 않고 이득만 주는 여지)가 좁다는 뜻으로 보인다. 파워커브류 신호 자체가 나쁜 게 아니라, **우리 모델은 이미 CatBoost가 ws10m/제곱/세제곱 등 원시 풍속 피처들로 파워커브에 가까운 관계를 스스로 근사하고 있어서, 굳이 명시적 파워커브 피처가 필요 없는 지점까지 이미 와 있다**는 해석이 지금까지 6번의 실패 패턴과 가장 잘 들어맞는다.

## 18. MLP 하이퍼파라미터 재검토

**배경**: 17절(SCADA 다중 NWP 소스 회귀 편향보정 + 파워커브)이 이동평균까지 포함해 6번째로 실패하면서, 이 파이프라인(50개 curated 피처 + τ=0.70 분위수회귀 + actual 가중)에서 피처 엔지니어링 레버는 소진된 것으로 판단했다. 지금까지 유일하게 진짜로 성공한 레버는 14절(MLP + 미분 가능한 FICR 손실)이었다 — CatBoost 단독(0.6213) 대비 CatBoost+MLP 블렌드가 CV +0.0082(표준편차의 6.2배), 실제 리더보드에서도 +0.0030 개선됐다(2차 제출 0.625596). 민석님이 이 레버를 더 파보기로 결정 — MLP 구조(은닉층 폭·깊이)·dropout·학습률을 우리 데이터로 직접 재검토한다.

**주의(외부 사례)**: 외부 문서(`phase8_accuracy_ceiling_and_affine.md` §2-2)에 따르면, 외부도 MLP 하이퍼파라미터를 Optuna로 튜닝했다가 **2023년 4-fold 교차검증에서는 +0.0119(t=4.90, 강한 신호)가 나왔지만, 실제 2024년 홀드아웃에서는 오히려 -0.0045로 역행**했다. 강한 정규화(dropout 0.40)가 데이터가 적은 fold에서 과최적화됐던 것으로 추정된다. **이번엔 같은 함정에 빠지지 않도록 세 가지 안전장치를 둔다**:
1. 3-fold 교차검증 평균뿐 아니라 **fold3(2024년 하반기, 가장 미래 방향)을 항상 따로 확인**한다 — 평균은 좋아지는데 fold3만 나빠지면 위험 신호로 본다
2. 최종 후보는 반드시 seed 3개(42/7/2024) 재검증을 거친 뒤에만 채택한다(11~17절과 같은 원칙)
3. dropout 탐색 범위를 외부가 과최적화를 겪은 0.40보다 낮게(0.05~0.35) 잡는다

**이번 절의 의존성**: 0~4절(셋업) + 9절(`to_long_ext`) + 14절(torch 임포트, `build_mlp_features`, `mlp_nn` 모듈, `best_T_soft`) — 14절을 실행한 상태에서 이어서 진행한다.

*(이 절은 실패/미채택으로 결론나 코드는 삭제하고 마크다운 기록만 남겼다 — 실행 결과는 각 소절 설명과 마지막 종합 해석, `reports/05_tuning.md`를 참고.)*

### 18-1. baseline 재확인

14절에서 확정한 현재 구조(은닉층 256-256, dropout 0.15, 학습률 1e-3, `T_soft`=`best_T_soft`)로 3-fold CV를 다시 찍어 비교 기준을 만든다. 14-6의 seed 재검증 평균(0.6256, 표준편차 0.0006)과 비슷한 크기가 나오는지 확인한다(단일 seed라 완전히 같지는 않을 수 있음).

### 18-2. 구조(은닉층 폭·깊이) 스윕

은닉층 폭·깊이 조합 4가지를 비교한다. dropout=0.15, `T_soft`=`best_T_soft`는 고정. **fold3(2024년 하반기, 가장 미래 방향) 점수를 별도 컬럼으로 남겨 평균만 보고 판단하지 않는다.**

**중간 점검(fold3 우선)**: 3-fold 평균으로는 `(256, 256, 256)`이 1위(0.6281)지만, **fold3(2024년 하반기, 가장 미래 방향)만 보면 `(256, 256)`이 압도적으로 1위(0.6538)** — 2위인 `(256, 128)`(0.6509)보다도 0.003 가까이 높고, 평균 1위였던 `(256, 256, 256)`의 fold3(0.6498)보다도 높다. 평균과 fold3이 가리키는 방향이 갈리므로, 18절 도입부에서 세운 원칙("fold3이 baseline보다 나빠지면 위험 신호로 본다")대로 **평균이 아니라 fold3을 우선해 구조를 `(256, 256)`으로 고정**한다. 이후 dropout·학습률 탐색은 이 구조를 기준으로 진행한다.

*(참고: 이 판단은 아래 코드로 `best_hidden`을 직접 덮어써서 반영한다 — 18-3·18-4는 코드 변경 없이 이 값을 그대로 이어받는다. 이전에 `(256,256,256)` 기준으로 18-3·18-4를 이미 실행하셨다면, 이 셀 이후로 18-3·18-4·18-5·18-6·18-7을 다시 실행해야 결과가 정확하다.)*

### 18-3. dropout 스윕

18-2에서 찾은 최적 구조(`best_hidden`)를 고정하고 dropout을 바꿔가며 비교한다. 외부가 과도한 dropout(0.40)에서 홀드아웃 역행을 겪었다는 점을 참고해 범위를 0.05~0.35로 잡는다.

### 18-4. 학습률 스윕

14절의 `train_mlp_fold`/`cv_score_mlp`는 학습률(`lr`)을 바꿀 수 있게 노출돼 있지 않아서, 이 절에서 `lr`/`weight_decay` 인자를 추가한 버전으로 다시 정의한다(이후 셀에서도 이 버전을 쓴다 — 14절의 것을 덮어쓰지만 인자를 안 주면 동작이 같다). 18-2·18-3에서 찾은 최적 구조·dropout을 고정하고 학습률만 바꿔 비교한다.

### 18-5. 최종 후보 정리 — baseline과 나란히 비교

18-2~18-4에서 찾은 최적 구조·dropout·학습률 조합을 baseline과 나란히 놓고 3-fold 평균과 fold3을 함께 확인한다. **fold3이 baseline보다 나빠지면(평균이 좋아 보여도) 위험 신호로 본다** — 외부 사례(홀드아웃 역행)를 재현하지 않기 위한 최소 안전장치다.

### 18-6. seed 재검증 (조건부)

11~17절과 같은 원칙에 fold3 안전장치를 더한다: 최고 후보가 baseline을 못 넘거나, 넘었어도 fold3이 baseline보다 나빠지면 채택하지 않는다. 두 조건을 다 통과했을 때만 seed 3개(42/7/2024)로 재검증한다.

### 18-7. CatBoost+MLP 블렌드 가중치 재탐색 (조건부)

18-6에서 새 MLP가 표준편차 이상으로 확인됐을 때만(15-5와 같은 원칙) 14-7·14-8과 같은 절차를 새 하이퍼파라미터로 반복한다 — MLP 자체가 좋아졌으니 블렌드 가중치도 다시 잡아야 할 수 있다.

### 18-8. 종합 해석

**18-2, 구조 스윕(핵심 발견)**: 3-fold 평균으로는 `(256,256,256)`이 1위(0.6281)였지만, **fold3(2024년 하반기, 가장 미래 방향)만 보면 `(256,256)`이 압도적 1위(0.6538)** -- 평균과 fold3이 가리키는 방향이 갈렸다. 18절 도입부에서 세운 원칙("fold3이 baseline보다 나빠지면 위험 신호")대로 평균 대신 fold3을 우선해 구조를 `(256,256)`(=현재 14절 채택 구조)으로 고정했다.

**18-3·18-4, dropout·학습률 재탐색(구조 고정 후)**: `(256,256)`을 고정하고 다시 탐색하니, **dropout=0.15·lr=1e-3(둘 다 현재 기본값)가 평균·fold3 양쪽 모두에서 1위**로 나왔다 -- 테스트한 다른 모든 dropout(0.01~0.35)·학습률(5e-4~3e-3) 값이 두 지표 모두에서 기본값보다 낮았다.

**18-5, 최종 비교**: baseline(256-256, drop0.15, lr1e-3)과 순차탐색 최적 조합이 **완전히 같은 설정으로 수렴**했다(평균 0.6265, fold3 0.6538, 개선폭 정확히 +0.0000). 이번엔 자기 자신과 비교한 게 아니라, 구조·dropout·학습률을 각각 독립적으로 탐색한 결과가 우연히 현재 설정과 일치한 것이다.

**18-6·18-7**: 개선이 없으니 자동으로 seed 재검증·블렌드 재탐색 모두 생략됨.

**결론**: **14절에서 채택한 MLP 하이퍼파라미터(은닉층 256-256, dropout 0.15, 학습률 1e-3)는 이미 우리 데이터·검증 구조에서 국소 최적점에 가깝다.** 구조를 넓히거나(384) 깊게(3층) 만들거나, dropout을 올리거나 내리거나, 학습률을 바꿔봐도 평균·fold3 어느 쪽으로도 더 나아지지 않았다. 중간에 평균만 보고 `(256,256,256)`을 골랐다면 fold3에서 역행하는 선택을 했을 뻔했다 -- 외부가 실제로 겪은 함정(CV는 좋아지고 홀드아웃은 나빠짐)을 이번엔 fold3 병행 확인으로 미리 걸러낸 셈이다.

**So-what**: 피처 엔지니어링(11~17절, 6연패)에 이어 MLP 하이퍼파라미터 탐색(18절)도 개선을 못 찾았다. 다만 이번 결과는 "실패"라기보다 "이미 잘 맞춰져 있었다는 확인"에 가깝다 -- 현재 확정 모델(CatBoost+MLP 그룹별 블렌드, 2차 제출 리더보드 0.625596)이 하이퍼파라미터 측면에서 쉽게 남겨둔 개선분이 없다는 뜻이다. 남은 후보(weight_decay 미탐색, 3번째 모델 블렌드, seed 앙상블 크기 조정 등)는 기대 효과 대비 투자가 더 크므로, 다음 방향은 민석님과 논의 필요.